# অলীকবচন — Phase A TRAINING notebook
### Replicates the exact feature implementation and architecture of the frozen inference notebook, but as the training/calibration phase.

**Purpose.** The inference notebook runs on a *frozen* `frozen_lr.pkl` and never re-fits anything. This notebook is the missing other half: it fits the per-regime logistic regressions from scratch on `grounded_augmented_v5_small.json` (the 896-row Phase A augmented set), using the identical deterministic feature layer, NLI scorer, and Qwen3-14B-AWQ judge as the inference notebook, and produces:
- `frozen_lr.pkl` — a drop-in replacement for the inference notebook's frozen pickle to save training time for our inference notebook
- `phase_a_benchmark_table.csv` — OOF F1/accuracy/macro-F1 for every model tier (standard, det+NLI, judge-only, final routed)
- `phase_a_confusion_matrix.csv` — OOF confusion matrix for the final routed+hard-rule prediction
- `LR_coefficients_grounded.csv` — full standardized LR coefficient vector
- `phase_a_per_pattern_report.csv` — OOF detection rate broken out by augmentation pattern (Track 1 + Track 2)
- `phase_a_feature_activation_examples.csv` — worked numeric examples for the patterns discussed in the paper

**Model note.** This run uses **Qwen3-14B-AWQ** as the judge (the inference notebook's current model), not the earlier Qwen2.5 build used before the final freeze — feature values and OOF numbers from this run are therefore the correct ones to report for the current, deployed architecture.

**Environment.** Kaggle, 2×T4 GPUs, same attached-dataset contract as the inference notebook: the Qwen3-14B-AWQ and mDeBERTa-v3-base-xnli model directories must be attached as Kaggle "models" datasets, and `grounded_augmented_v5_small.json` must be attached as a Kaggle dataset input. No Hugging Face fallback exists anywhere in this pipeline by design — a missing model/data attachment is a hard `FileNotFoundError`, not a silent degrade.



## 0 · Configuration

In [ ]:
import sys, subprocess, os
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")

def _run(cmd, check=False):
    print("[install] $", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"[install]   FAILED (exit {r.returncode})")
        print(r.stdout[-2000:]); print(r.stderr[-2000:])
        if check:
            raise RuntimeError(f"pip command failed (exit {r.returncode}): {' '.join(cmd)}")
    else:
        print("[install]   ok")
    return r

INTERNET_ON = False   # <-- flip this for the real (offline) submission run
WHEELS = "/kaggle/input/datasets/raufrahim/offline-wheels/offline_wheels"   # match your dataset's mount folder name

if INTERNET_ON:
    _run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
          "torch", "torchvision", "torchaudio", "transformers", "tokenizers",
          "numpy", "accelerate", "bitsandbytes", "vllm"])
    _run([sys.executable, "-m", "pip", "install", "-q", "-U", "vllm==0.8.5"], check=True)
    _run([sys.executable, "-m", "pip", "install", "-q", "-U",
          "transformers==4.51.3", "tokenizers==0.21.1"], check=True)
    _run([sys.executable, "-m", "pip", "install", "-q", "-U",
          "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
          "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)
    _run([sys.executable, "-m", "pip", "install", "-q", "numpy==2.0.0"], check=True)
    _run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", "pandas"], check=True)
    _run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "pynvml"])
    _run([sys.executable, "-m", "pip", "install", "-q", "nvidia-ml-py", "pygments", "pyarrow"], check=True)
    _run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers==2.7.0"], check=True)
    _run([sys.executable, "-m", "pip", "install", "-q", "FlagEmbedding==1.2.11", "bm25s"], check=True)
    _run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers==4.51.3"], check=True)
    _run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "sympy", "huggingface_hub", "hf_transfer"], check=True)
else:
    # --no-index: never touch PyPI. --find-links: resolve every package (and its
    # transitive deps) from the pre-built wheels in the attached dataset instead.
    _run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", WHEELS,
          "vllm==0.8.5", "transformers==4.51.3", "tokenizers==0.21.1",
          "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
          "numpy==2.0.0", "pandas", "nvidia-ml-py", "pygments", "pyarrow",
          "sentence-transformers==2.7.0", "FlagEmbedding==1.2.11", "bm25s",
          "rapidfuzz", "sympy", "huggingface_hub", "hf_transfer"], check=True)

print("[install] PINNED Qwen3-compatible stack installed on disk.")

In [ ]:
import os, sys, json, re, gc, math, time, random, glob, warnings, unicodedata
from dataclasses import dataclass, field
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
SEED = 20260702
random.seed(SEED); np.random.seed(SEED)

class CFG:
    # ------------------------------------------------------------------ paths
    # TRAINING notebook: the only data source is the Phase A augmented set.
    # There is no test_df / submission step here -- this notebook exists to
    # produce frozen_lr.pkl + the paper's numeric reports (benchmarks, OOF,
    # coefficients, per-pattern hit rates, feature-activation examples).
    AUGMENTED_TRAINING_JSON = "grounded_augmented_v5_small.json"

    # ------------------------------------------------------- component toggles
    USE_NLI        = True
    USE_LLM_JUDGE  = True

    # ------------------------------------------------------------------ models
    # Same model choices as the frozen inference notebook -- Qwen3 (not the
    # earlier Qwen2.5 this pipeline used pre-freeze), so the frozen_lr.pkl this
    # notebook produces is trained on features computed by the SAME judge that
    # will score it at inference time. All models are pulled from the attached
    # Kaggle "models" dataset ONLY, via resolve_local_model_dir(): no Hugging
    # Face id, no fallback list -- a missing/failed model load is a hard error.
    NLI_MODEL_NAME = "mDeBERTa-v3-base-xnli-multilingual-nli-2"
    JUDGE_MODEL_NAME = "Qwen3-14B-AWQ"
    LOCAL_MODEL_DIRS = ["/kaggle/input"]
    MAX_CTX_TOKENS   = 1024
    BATCH_NLI        = 32

    # ---------------------------------------------------------------- logging
    QWEN_LOG_FILE    = "qwen_responses_train.json"
    ARCH_LOG_FILE    = "architecture_fallbacks_train.json"

    # ---------------------------------- Chain-of-Thought judge (n3 integration)
    USE_COT_JUDGE      = True
    COT_N_FEWSHOT      = 13
    COT_MAX_NEW_TOKENS = 192
    COT_MAX_CTX_CHARS  = 6500
    JUDGE_MAX_PARAMS_B = 14

    # ---------------------------------- vLLM judge backend (2xT4 tensor-parallel)
    VLLM_DTYPE         = "float16"
    VLLM_GPU_UTIL      = 0.75
    VLLM_MAX_MODEL_LEN = 8000
    VLLM_QUANTIZATION  = "awq"
    VLLM_ENFORCE_EAGER = True
    VLLM_LOGPROB_BLOCK = 64
    COT_VLLM_BLOCK     = 128

    # -------------------------------------------------------------- ensembling
    N_FOLDS               = 5
    RUNTIME_BUDGET_HOURS  = 8.5
    HARD_RULES            = True

T0 = time.time()
def elapsed_h(): return (time.time() - T0) / 3600.0
def budget_ok(): return elapsed_h() < CFG.RUNTIME_BUDGET_HOURS

FALLBACK_LOG = []
def note_fallback(component: str, msg: str):
    line = f"[FALLBACK] {component}: {msg}"
    FALLBACK_LOG.append(line)
    print("\n" + "!" * 78 + f"\n{line}\n" + "!" * 78)

ARCH = {"nli_model": None, "judge_model": None, "judge_quant": None}

ON_KAGGLE = os.path.exists("/kaggle/input")
OUTPUT_DIR = "/kaggle/working" if ON_KAGGLE else "."
print(f"Kaggle environment: {ON_KAGGLE} | output dir: {OUTPUT_DIR}")


## 1 · Data loading
Loads `grounded_augmented_v5_small.json` directly (the same file `CFG.AUGMENTED_TRAINING_JSON` points the inference notebook at) and builds the `seed_group` column `StratifiedGroupKFold` needs downstream, so that an original seed row and its augmented siblings never split across train/validation folds within the same OOF fit.

In [ ]:
def _find(fname: str) -> str:
    if os.path.exists(fname):
        return fname
    hits = sorted(glob.glob(f"/kaggle/input/**/{fname}", recursive=True)) if ON_KAGGLE else []
    if hits:
        return hits[0]
    raise FileNotFoundError(fname)

def _clean(df: pd.DataFrame, has_label: bool) -> pd.DataFrame:
    df = df.copy()
    for col in ("context", "prompt_bn", "response_bn"):
        df[col] = df[col].apply(lambda x: unicodedata.normalize("NFC", str(x)) if pd.notna(x) else "")
    df["is_closed_book"] = df["context"].str.strip().isin(["[NULL]", "", "nan", "None"])
    df.loc[df["is_closed_book"], "context"] = ""
    if has_label:
        df["label"] = df["label"].astype(int)
    if "id" not in df.columns:
        df["id"] = df.index.astype(str)
    return df

# ---------------------------------------------------------------------------
# Load the Phase A augmented set (grounded_augmented_v5_small.json, 896 rows)
# -- fails loud (FileNotFoundError) if missing, no fallback to any smaller/
# unaudited set, matching the inference notebook's own loading contract.
# ---------------------------------------------------------------------------
aug_path = _find(CFG.AUGMENTED_TRAINING_JSON)
with open(aug_path, encoding="utf-8") as f:
    aug_rows = json.load(f)
aug_df = pd.DataFrame(aug_rows)
aug_df = _clean(aug_df, has_label=True)

# All rows in this file are grounded by construction (Phase A only), but keep
# the same is_closed_book scoping the inference notebook applies, as a guard
# against a stray malformed row rather than assuming it.
if aug_df["is_closed_book"].any():
    n_bad = int(aug_df["is_closed_book"].sum())
    print(f"[WARNING] {n_bad} rows in {CFG.AUGMENTED_TRAINING_JSON} have empty/[NULL] context "
          f"-- Phase A is grounded-only; these will still be scored but are unexpected here.")

# seed_group: rows sharing a "seed_XXXX" prefix are near-duplicate-context siblings
# (an original seed + its LLM-generated / procedural / hand-crafted variants) and
# MUST stay together in CV folds -- this is what StratifiedGroupKFold uses below.
# Standalone hand-crafted pattern rows (pat*_trap/ctrl, genpos_*) have no seed
# siblings, so each becomes its own singleton group.
_seed_prefix = aug_df["id"].astype(str).str.extract(r'^(seed_\d+)')[0]
aug_df["seed_group"] = _seed_prefix.fillna(aug_df["id"].astype(str))

n_seed_groups = aug_df.loc[_seed_prefix.notna(), "seed_group"].nunique()
n_standalone  = int(_seed_prefix.isna().sum())
print(f"[Phase A training] {len(aug_df)} rows loaded from {CFG.AUGMENTED_TRAINING_JSON}")
print(f"  seed-groups with siblings: {n_seed_groups} | standalone hand-crafted rows: {n_standalone}")
print(f"  label balance: faithful={int((aug_df.label == 1).sum())} hallucinated={int((aug_df.label == 0).sum())}")
if "pattern" in aug_df.columns:
    n_patterned = int(aug_df["pattern"].notna().sum())
    print(f"  rows with a named augmentation pattern (Track 2, hand-crafted): {n_patterned}")

train_df = aug_df.reset_index(drop=True)


## 2 · Bengali normalization & numeric semantics
Verbatim from the inference notebook — surface-form-invariant number parsing (digits↔words, Bengali↔ASCII numerals, ZWJ/ZWNJ, punctuation) that every downstream feature depends on. No sample/test split logic exists in this cell to begin with (pure function/constant definitions), so nothing needed adapting.

In [ ]:
BN_DIGITS = "০১২৩৪৫৬৭৮৯"
_BN2EN = str.maketrans(BN_DIGITS, "0123456789")
_BENGALI_WORD = re.compile(r"[\u0980-\u09FF]+")
_PUNCT = re.compile(r"""[।,;:!?"'()\[\]{}«»“”‘’–—\-.·]""")

# Bengali number words. Bengali uses irregular composite numerals (not
# tens+units like English), so 0-100 needs an explicit table; we include the
# high-frequency entries plus every value observed in the data, and all
# multipliers of the South-Asian system (হাজার/লক্ষ/কোটি).
BN_ONES = {
    "শূন্য":0,"এক":1,"দুই":2,"দু":2,"তিন":3,"চার":4,"পাঁচ":5,"ছয়":6,"ছয়":6,"সাত":7,"আট":8,"নয়":9,"নয়":9,
    "দশ":10,"এগারো":11,"এগার":11,"বারো":12,"বার":12,"তেরো":13,"তের":13,"চৌদ্দ":14,"পনেরো":15,"পনের":15,
    "ষোল":16,"ষোলো":16,"সতেরো":17,"সতের":17,"আঠারো":18,"আঠার":18,"উনিশ":19,"বিশ":20,"কুড়ি":20,
    "একুশ":21,"বাইশ":22,"তেইশ":23,"চব্বিশ":24,"পঁচিশ":25,"ছাব্বিশ":26,"সাতাশ":27,"আটাশ":28,"ঊনত্রিশ":29,
    "ত্রিশ":30,"একত্রিশ":31,"বত্রিশ":32,"তেত্রিশ":33,"চৌত্রিশ":34,"পঁয়ত্রিশ":35,"ছত্রিশ":36,"সাঁইত্রিশ":37,
    "আটত্রিশ":38,"ঊনচল্লিশ":39,"চল্লিশ":40,"একচল্লিশ":41,"বিয়াল্লিশ":42,"তেতাল্লিশ":43,"চুয়াল্লিশ":44,
    "পঁয়তাল্লিশ":45,"ছেচল্লিশ":46,"সাতচল্লিশ":47,"আটচল্লিশ":48,"ঊনপঞ্চাশ":49,"পঞ্চাশ":50,
    "একান্ন":51,"বায়ান্ন":52,"তিপ্পান্ন":53,"চুয়ান্ন":54,"পঞ্চান্ন":55,"ছাপ্পান্ন":56,"সাতান্ন":57,
    "আটান্ন":58,"ঊনষাট":59,"ষাট":60,"একষট্টি":61,"বাষট্টি":62,"তেষট্টি":63,"চৌষট্টি":64,"পঁয়ষট্টি":65,
    "ছেষট্টি":66,"সাতষট্টি":67,"আটষট্টি":68,"ঊনসত্তর":69,"সত্তর":70,"একাত্তর":71,"বাহাত্তর":72,
    "তিয়াত্তর":73,"চুয়াত্তর":74,"পঁচাত্তর":75,"ছিয়াত্তর":76,"সাতাত্তর":77,"আটাত্তর":78,"ঊনআশি":79,
    "আশি":80,"একাশি":81,"বিরাশি":82,"তিরাশি":83,"চুরাশি":84,"পঁচাশি":85,"ছিয়াশি":86,"সাতাশি":87,
    "অষ্টাশি":88,"ঊননব্বই":89,"নব্বই":90,"একানব্বই":91,"বিরানব্বই":92,"তিরানব্বই":93,"চুরানব্বই":94,
    "পঁচানব্বই":95,"ছিয়ানব্বই":96,"সাতানব্বই":97,"আটানব্বই":98,"নিরানব্বই":99,"একশ":100,"একশত":100,
}
# NB: "শত" must live ONLY in BN_MULT — it is a multiplier ("দুই শত" = 200);
# putting it in BN_ONES turns "দুই শত" into 2+100=102.
BN_MULT = {"শ":100,"শত":100,"হাজার":1000,"লক্ষ":100000,"লাখ":100000,"কোটি":10000000,"মিলিয়ন":1000000,"বিলিয়ন":1000000000}
# "আঠারশ" = আঠারো + শ fused (eighteen-hundred); handle fused <ones>+শ tokens generically.
_FUSED_SHO = re.compile(r"^(.+?)শ$")

# ---- classifier/counter-suffix-fused number words (e.g. "তিনটি"=three
# [pieces], "পাঁচটি"=five [pieces], "তিনবার"=three times, "দুটি"=two [pieces]).
# Bengali attaches a classifier/counter word directly to the numeral with no
# space, so the bare BN_ONES lookup above never matched these -- confirmed
# on real test-set rows (id726 "তিনবার", id1255 "তিনটি", id1324 "পাঁচটি")
# where a CORRECT answer's number-word silently failed to parse, making
# numbers_of() return an empty set for that side and triggering a false
# num_contra/n3_ground mismatch downstream. Longest-suffix-first, same
# principle as _BN_SUFFIXES in the deterministic-feature cell.
#
# "এক" + article-like suffix (একটি/একটা/একজন/...) is deliberately EXCLUDED:
# এক+টি/টা/জন is overwhelmingly the Bengali indefinite article ("a"/"an"),
# not a literal count of one -- resolving it would inject a bogus 1.0 into
# numbers_of() for one of the most common word patterns in the language.
# "একবার" (lit. "one time" = once) is the one exception kept: বার here is a
# genuine multiplicative counter, not an article, so এক+বার still resolves.
_ARTICLE_LIKE_SUFFIXES = ("টির", "টিতে", "টি", "টার", "টা", "জনের", "জন", "খানা", "খানি")
_COUNTER_SUFFIXES = _ARTICLE_LIKE_SUFFIXES + ("বার",)
_COUNTER_SUFFIX_RE = re.compile("(" + "|".join(sorted(_COUNTER_SUFFIXES, key=len, reverse=True)) + ")$")

def _strip_counter_suffix(w: str):
    """If w is <number-word>+<classifier/counter suffix> fused with no space,
    return (base_word, suffix) with the suffix removed; else (w, None)."""
    m = _COUNTER_SUFFIX_RE.search(w)
    if not m:
        return w, None
    suffix = m.group(1)
    base = w[:m.start()]
    if base == "এক" and suffix in _ARTICLE_LIKE_SUFFIXES:
        return w, None   # indefinite article, not a literal "one" -- don't resolve
    if base and (base in BN_ONES or _FUSED_SHO.match(base)):
        return base, suffix
    return w, None

def bn_words_to_numbers(text: str) -> set:
    """Extract integers spelled as Bengali number words. Returns a set of ints."""
    t = re.sub(r"[’'`]", "", str(text))
    vals, cur, total, seen = set(), 0, 0, False
    def flush():
        nonlocal cur, total, seen
        if seen and (total + cur) > 0:
            vals.add(total + cur)
        cur, total, seen = 0, 0, False
    for w in _BENGALI_WORD.findall(t):
        if w in BN_ONES:
            cur += BN_ONES[w]; seen = True
        elif w in BN_MULT:
            m = BN_MULT[w]
            if cur == 0: cur = 1
            if m >= 1000: total += cur * m; cur = 0
            else: cur *= m
            seen = True
        else:
            fm = _FUSED_SHO.match(w)
            if fm and fm.group(1) in BN_ONES:      # আঠারশ, উনিশশ, ...
                total += BN_ONES[fm.group(1)] * 100; seen = True
            else:
                base, suffix = _strip_counter_suffix(w)
                if suffix is not None and base in BN_ONES:
                    cur += BN_ONES[base]; seen = True
                elif suffix is not None and _FUSED_SHO.match(base) and _FUSED_SHO.match(base).group(1) in BN_ONES:
                    total += BN_ONES[_FUSED_SHO.match(base).group(1)] * 100; seen = True
                else:
                    flush()
    flush()
    return vals

def normalize(s: str) -> str:
    s = unicodedata.normalize("NFC", str(s)).translate(_BN2EN)
    s = s.replace("\u200c", "").replace("\u200d", "")
    # FIX (round-6, real-run df_analysis sweep): comma-grouped thousands
    # separators ("৫০,০০০" = 50,000; Indian-style "১৫,০০,০০০" = 1,500,000)
    # were previously fragmented by the generic punctuation pass below --
    # numbers_of("৫০,০০০") returned {50.0, 0.0}, NEITHER of which is the
    # true value. Strip a comma sitting directly between two digits before
    # anything else touches it (same principle as the decimal-point
    # protection just below, but here the separator is removed entirely,
    # not restored). Confirmed in 18 real test-set responses / 65 contexts.
    s = re.sub(r"(?<=\d),(?=\d)", "", s)
    s = re.sub(r"(?<=\d)\.(?=\d)", "\u0001", s)   # protect decimal points (৭.৬৩)
    s = _PUNCT.sub(" ", s).replace("\u0001", ".")
    return re.sub(r"\s+", " ", s).strip().lower()

def tokens(s: str) -> set:
    return set(normalize(s).split())

_NUM = re.compile(r"\d+(?:\.\d+)?")

# FIX (round-6): digit + Bengali multiplier-word combos ("১২ লক্ষ" = 1,200,000,
# "৬ লক্ষ ৭৫ হাজার" = 675,000) were previously mis-parsed -- bn_words_to_numbers()
# only recognizes spelled-out WORD numbers before a multiplier; a bare DIGIT
# preceding "লক্ষ"/"কোটি"/etc. made it silently default to treating the
# digit as 1 (observed: "১২ লক্ষ" -> {100000, 12}, dropping the real value
# entirely). Confirmed in 9 real test-set responses / 37 contexts. This
# explicitly matches <digit><multiplier-word> spans, computes digit*multiplier
# directly, chain-sums consecutive spans in the same expression (for "X লক্ষ
# Y হাজার" style compounds), and the consumed digit spans are masked out of
# the string before the generic digit/word passes below so they don't also
# emit spurious fragments.
_DIGIT_MULT_RE = re.compile(r"(\d+(?:\.\d+)?)\s*(লক্ষ|লাখ|কোটি|হাজার|মিলিয়ন|বিলিয়ন|শত)(?!\w)")
# FIX (round-7, wide-verification sweep on v6 diagnostics): bare "শ" (100) was
# dropped from this list. Python's \w does NOT match Bengali dependent vowel
# signs (matras) -- confirmed: re.match(r"\w", "ে") is None -- so the (?!\w)
# guard failed to block "শ" from matching inside "২৬শে" (a date-ordinal
# suffix, "the 26th", extremely common in Bengali date-writing), producing
# 26*100=2600 instead of leaving "26" alone. Real, confirmed regression:
# id1061's exact "২০০৩ সালের ২৬শে মার্চ" match was broken by this. "শত" (the
# full word) is unambiguous and kept; bare "শ" as a colloquial hundred-
# multiplier is rare enough that dropping it entirely is the safer trade.

# FIX (round-6): ordinal words ("তৃতীয়"=3rd, "নবম"=9th) were entirely
# invisible to numbers_of() before -- confirmed in 80 real test-set
# responses / 486 contexts containing one. Mapped to their rank value so a
# response like "নবম স্থান" can be numerically verified against context.
BN_ORDINALS = {"প্রথম":1,"দ্বিতীয়":2,"তৃতীয়":3,"চতুর্থ":4,"পঞ্চম":5,"ষষ্ঠ":6,"সপ্তম":7,
               "অষ্টম":8,"নবম":9,"দশম":10,"একাদশ":11,"দ্বাদশ":12,"ত্রয়োদশ":13,"চতুর্দশ":14,
               "পঞ্চদশ":15,"ষোড়শ":16,"সপ্তদশ":17,"অষ্টাদশ":18,"ঊনবিংশ":19,"বিংশ":20}

def numbers_of(s: str) -> set:
    """All numeric values in a string: digits (bn/en) + spelled Bengali words +
    digit-multiplier compounds + ordinal words."""
    ns = normalize(s)

    consumed_spans = []
    compound_vals = []
    running_sum, has_run = 0.0, False
    last_end = None
    for m in _DIGIT_MULT_RE.finditer(ns):
        val = float(m.group(1)) * BN_MULT[m.group(2)]
        if last_end is not None and ns[last_end:m.start()].strip() == "":
            running_sum += val
        else:
            if has_run:
                compound_vals.append(running_sum)
            running_sum = val
        has_run = True
        last_end = m.end()
        consumed_spans.append((m.start(), m.end()))
    if has_run:
        compound_vals.append(running_sum)

    masked = list(ns)
    for s0, e0 in consumed_spans:
        for i in range(s0, e0):
            masked[i] = " "
    masked_ns = "".join(masked)

    vals = set(float(x) for x in _NUM.findall(masked_ns))
    vals |= set(float(x) for x in bn_words_to_numbers(masked_ns))
    vals |= set(compound_vals)

    for tok in _BENGALI_WORD.findall(re.sub(r"[’'`]", "", str(s))):
        if tok in BN_ORDINALS:
            vals.add(float(BN_ORDINALS[tok]))

    return vals

# ---- numeralize() -- number-form normalization for exact_contain/
# fuzzy_contain specifically. normalize() already unifies bn/en DIGITS, but
# has no concept of spelled-out number WORDS ("আঠারশ বত্রিশ") -- so a
# correct answer spelled in words never string-matches the same number in
# digit form in the context, silently defeating exact_contain/fuzzy_contain
# even though numbers_of()/num_in_ctx already recognize them as equal
# semantically. numeralize() replaces maximal runs of digit/word number
# pieces with their combined digit value, so exact_contain/fuzzy_contain can
# see the equivalence too. Word-form pieces (and digit+multiplier-word
# combos like '৬ হাজার') compose additively/multiplicatively, matching how
# Bengali numbers are actually spoken/written. Two adjacent BARE DIGIT runs
# never merge (that's almost always two separate numbers, e.g. in a list or
# range, not one composed number) -- the one deliberate safety cutout.
_NUM_OR_WORD = re.compile(r"\d+(?:\.\d+)?|[\u0980-\u09FF]+")

def _numeralize_piece_kind(tok: str):
    if _NUM.fullmatch(tok):
        return "digit", float(tok)
    if tok in BN_ONES:
        return "one", BN_ONES[tok]
    if tok in BN_MULT:
        return "mult", BN_MULT[tok]
    fm = _FUSED_SHO.match(tok)
    if fm and fm.group(1) in BN_ONES:
        return "fused", BN_ONES[fm.group(1)] * 100
    return None, None

def numeralize(s: str) -> str:
    s = re.sub(r"[’'`]", "", str(s)).translate(_BN2EN)
    # SAME FIX as normalize() above, applied here too since numeralize() does
    # its own independent preprocessing (doesn't call normalize()) and won't
    # inherit that fix automatically. Without this, the "don't merge two bare
    # digit-groups" safety rule a few lines down (built to avoid merging
    # genuinely unrelated numbers like "১২ এবং ৩৪") has the side effect of
    # breaking legitimate comma-grouped numbers instead: "৫০,০০০" was
    # observed producing "50,0" (silently losing two zeros) since the two
    # digit-groups either side of the comma were correctly left unmerged by
    # that rule, but the comma itself was never actually stripped.
    s = re.sub(r"(?<=\d),(?=\d)", "", s)
    spans = [(m.group(0), m.start(), m.end()) for m in _NUM_OR_WORD.finditer(s)]
    out, pos, i, n = [], 0, 0, len(spans)
    while i < n:
        tok, start, end = spans[i]
        kind, val = _numeralize_piece_kind(tok)
        if kind is None:
            i += 1; continue
        run_start, run_end, j = start, end, i + 1
        cur, total, prev_kind = val, 0.0, kind
        while j < n:
            tok2, start2, end2 = spans[j]
            if s[run_end:start2].strip(" ,-") != "":
                break
            kind2, val2 = _numeralize_piece_kind(tok2)
            if kind2 is None:
                break
            if prev_kind == "digit" and kind2 == "digit":
                break   # two bare numbers in a row -- don't merge, likely unrelated
            if kind2 == "mult":
                if val2 >= 1000:
                    total += cur * val2; cur = 0
                else:
                    cur *= val2
            else:
                cur += val2
            run_end = end2; prev_kind = kind2; j += 1
        final_val = total + cur
        out.append(s[pos:run_start])
        out.append(str(int(final_val)) if float(final_val).is_integer() else str(final_val))
        pos = run_end
        i = j if j > i + 1 else i + 1
    out.append(s[pos:])
    return "".join(out)

# ---- unit tests (fail fast if the parser regresses) ------------------------
assert bn_words_to_numbers("দুই লক্ষ তেরো হাজার দুই শত জন") == {213200}
assert bn_words_to_numbers("আঠারশ' বত্রিশ সালে") == {1832}
assert bn_words_to_numbers("তিন ভাগে ভাগ") == {3}
assert 1937.0 in numbers_of("১৯৩৭ সালে")
assert normalize("৭.৬৩ ট্রিলিয়ন").startswith("7.63")
assert numeralize("আঠারশ' বত্রিশ সালে") == "1832 সালে"
assert numeralize("প্রায় ৬ হাজার") == "প্রায় 6000"
assert numeralize("৫ কোটি টাকা") == "50000000 টাকা"
assert numeralize("তালিকায় ১২ এবং ৩৪ আছে") == "তালিকায় 12 এবং 34 আছে"   # no false merge

# ---- classifier/counter-suffix-fused number word tests (real test-set false
# positives: id726 "তিনবার", id1255 "তিনটি", id1324 "পাঁচটি" -- see cell notes) ----
assert numbers_of("তিনবার জাতীয় পুরস্কার") == {3.0}
assert numbers_of("তিনটি পর্যায়ে ভাগ করা হয়") == {3.0}
assert numbers_of("পাঁচটি মহকুমা নিয়ে গঠিত") == {5.0}
assert numbers_of("দুটি বই") == {2.0}
assert numbers_of("দুইটি বই") == {2.0}
assert numbers_of("একবার দেখা হয়েছিল") == {1.0}          # "once" -- genuine literal one
assert 1.0 not in numbers_of("একটি বই পড়লাম")            # article ("a book"), NOT the number one
assert 1.0 not in numbers_of("একজন লোক এসেছিল")           # article ("a person"), NOT the number one
print("normalization unit tests passed ✓")

## 3 · Deterministic feature layer
Verbatim `det_features()` from the inference notebook (question-type regexes, containment/contradiction checks, negation/hedge/calendar/arithmetic/kinship features, relevant-sentence scoping, etc.) — the only change is the tail: `build_det()` is now called once, on `train_df`, instead of twice on `sample_df`/`test_df`.

In [ ]:
Q_NUMERIC  = re.compile(r"(কত|কবে|কোন সালে|কত সালে|সাল|তারিখ|সংখ্যা|বছর|শতাংশ|দৈর্ঘ্য|উচ্চতা|আয়তন|জনসংখ্যা)")
Q_PERSON   = re.compile(r"(কে |কে\?|কার |কাকে|কাদের|রচয়িতা|লেখক|প্রতিষ্ঠাতা|আবিষ্কার|উদ্ভাবন|নির্মাতা|পরিচালক)")
Q_PLACE    = re.compile(r"(কোথায়|কোন দেশ|কোন জেলা|কোন শহর|কোন অঞ্চল|রাজধানী|অবস্থিত)")
Q_MEANING  = re.compile(r"(অর্থ|ভাবার্থ|বাগধারা|মানে|বলতে কী|বলতে কি|পূর্ণরূপ|পুরো নাম|সংজ্ঞা)")
Q_MATH     = re.compile(r"(=|[+×÷*/^√]|সমীকরণ|মান কত|যোগফল|গুণফল|ভাগফল|অনুপাত|শতকরা|গড়|ক্ষেত্রফল|পরিসীমা|x²|x\*\*)")
Q_MCQ      = re.compile(r"(কোনটি|নিচের কোন|নীচের কোন|সঠিক উত্তর|ক\)|খ\)|গ\)|ঘ\))")
Q_YESNO    = re.compile(r"(কি\?|কী\?|কি না|is it|কি\s*$)")
ABSTAIN    = re.compile(r"(তথ্য নেই|জানা যায়নি|জানা নেই|উল্লেখ নেই|নির্দিষ্ট করে বলা|পাওয়া যায়নি|বলা সম্ভব নয়|উত্তর দেওয়া সম্ভব)")
BN_YES     = re.compile(r"^(হ্যাঁ|হ্যা|জি|জ্বি|জ্বী|অবশ্যই)")
BN_NO      = re.compile(r"^(না|নাহ|মোটেও না)")

# =============================================================================
# NEW CANDIDATE FEATURES (added for feature-selection EDA -- not yet validated,
# not yet in the master notebook). Each addresses a specific gap identified so
# far:
#   - content_cov / char3_contain: cov_tokens/fuzzy_contain undercredit answers
#     whose tokens carry a Bengali case/classifier suffix not present verbatim
#     in the context (e.g. "২২টি" vs context "২২") -- flagged as a known risk
#     in prior notes ("suffix-agglutination undercrediting"). content_cov uses
#     a crude suffix-stripped, stopword-filtered token set; char3_contain is a
#     character-trigram containment score, which is agnostic to token boundaries
#     entirely and catches partial morphological overlap neither cov_tokens
#     nor fuzzy_contain (substring-window based, not set-based) captures.
#   - num_extra_ratio: a continuous version of num_contra (which is a hard
#     0/1) -- what FRACTION of the answer's numbers are unsupported, not just
#     whether at least one is.
#   - resp_ctx_ratio / resp_q_ratio: relative length features (answer length
#     as a fraction of context/question length) rather than raw resp_len,
#     which the ablation showed to be a somewhat unstable/redundant signal.
#   - hedge_flag / multi_answer: Bengali hedging language ("সম্ভবত", "মনে হয়")
#     or an "X বা Y" multi-candidate answer are both patterns that plausibly
#     correlate with an uncertain/hallucinated answer.
#   - year_implausible: a response's only number looks like a year but falls
#     outside a sane historical range -- a cheap sanity check independent of
#     context-matching (catches invented dates even when no context exists).
# =============================================================================
_BN_STOPWORDS = {
    "এবং","ও","বা","অথবা","কিন্তু","তবে","যে","যা","এই","ঐ","সে","তার","তাদের","এর","এটি","এটা",
    "একটি","একটা","হয়","হয়েছে","ছিল","আছে","করে","করেছে","থেকে","জন্য","সাথে","পর","আগে","মধ্যে",
    "উপর","নিচে","কে","কি","না","হ্যাঁ","তে","য়","র","এ","ই","টি","টা","গুলো","গুলি","দের",
}
# longest-suffix-first so e.g. "গুলোতে" doesn't only strip as "তে"
_BN_SUFFIXES = ("গুলোর","গুলোতে","গুলো","গুলির","গুলি","দেরকে","দের","টির","টিতে","টি","টার","টা","কে","তে","\u09c7\u09b0","র")
# NOTE (validated against real rows, e.g. id195 "উপন্যাসের"): the possessive suffix as it
# actually attaches to a consonant is the DEPENDENT e-kar vowel sign (U+09C7) + র (ের),
# not the independent এ (U+098F) that was here before -- that older entry silently never
# matched real text. It must also come BEFORE the bare "র" entry, not after: _strip_suffix
# returns on first match, and "র" alone would otherwise always win first, per the same
# longest-suffix-first ordering principle already used for গুলো/গুলোতে above.

def _strip_suffix(tok: str) -> str:
    """Crude Bengali suffix stripper -- NOT a real stemmer, just enough to stop
    a bare classifier/case suffix from silently breaking a token match."""
    for suf in _BN_SUFFIXES:
        if tok.endswith(suf) and len(tok) > len(suf) + 1:
            return tok[: -len(suf)]
    return tok

def content_tokens(s: str) -> set:
    """Stopword-filtered, suffix-stripped token set -- complements tokens()/
    cov_tokens, which count every whitespace token including particles and
    un-stemmed inflected forms."""
    return {_strip_suffix(t) for t in tokens(s) if t not in _BN_STOPWORDS and len(t) > 1}

def char_ngrams(s: str, n: int = 3) -> set:
    s = s.replace(" ", "")
    if len(s) < n:
        return {s} if s else set()
    return {s[i:i + n] for i in range(len(s) - n + 1)}

def char_ngram_containment(a: str, b: str, n: int = 3) -> float:
    """Fraction of ANSWER's character trigrams found in CONTEXT -- containment
    (denominator = |A|), not symmetric set-overlap. A symmetric measure was
    tried first and inverted the signal: context is usually far longer than
    the answer, so its trigram set dominates the denominator regardless of
    match quality. Containment is set-based and boundary-agnostic (unlike
    fuzzy_contain, which is a best-window substring similarity), so it still
    adds a distinct signal on top of fuzzy_contain/cov_tokens."""
    A, B = char_ngrams(a, n), char_ngrams(b, n)
    if not A or not B:
        return 0.0
    return len(A & B) / len(A)

HEDGE_RE    = re.compile(r"(সম্ভবত|মনে হয়|প্রায়|আনুমানিক|হয়তো|যদিও|বলা যায়)")
MULTI_ANS_RE = re.compile(r"(\sবা\s|\sঅথবা\s|/)")

def _year_plausible(nums: set) -> int:
    """1 if every year-shaped number (4 digits, treated as 1000-9999) in the
    answer falls in a sane historical range (1000-2035); 0 if any looks like
    an invented/bogus year; 0 (not applicable) if no year-shaped number."""
    years = [x for x in nums if 1000 <= x <= 9999]
    if not years:
        return 0
    return int(all(1000 <= y <= 2035 for y in years))

# =============================================================================
# NEW (Category A/B fix, this round): two features targeting the two
# recurring failure clusters from the round-6/7 analysis -- both computed
# from det+question signals only, no judge dependency.
#
# relevant_context_for_question(): the containment-override cluster (ids 3,
# 53, 79, 217) is a wrong answer coincidentally matching text ELSEWHERE in a
# longer passage. Restricts exact_contain/fuzzy_contain to the sentence(s)
# most relevant to the QUESTION (by content-word overlap, generic QA
# scaffold words excluded so they don't dominate the count) instead of the
# whole passage. Validated against real failure rows: fixes id53 cleanly
# (the wrong "reserves" sentence and the right "demand" sentence are
# genuinely separate, non-tied). Does NOT fix id3 (the wrong answer is a
# category mismatch -- a film name for a "what year" question -- sitting
# INSIDE the correctly-identified relevant sentence itself; that's a
# different problem, already attempted separately via an entity-type prompt
# instruction, which caused net regressions and was reverted). Only
# partially helps id79 (its two candidate sentences can still tie on
# generic overlap even after scaffold-word exclusion, since "ম্যাচ" (match)
# and "টেস্ট" (Test) are both genuinely cricket-relevant vocabulary, not
# generic scaffolding, and only phrase-level matching would fully resolve
# this). Falls back to the FULL context (today's exact behavior) whenever no
# sentence clears the relevance bar, so rows this heuristic doesn't apply to
# are completely unaffected.
_SENT_SPLIT_RE = re.compile(r'(?<=[।.!?])(?:\[\d+\])?\s+')
# NOTE (validated against real rows, e.g. id195): Wikipedia-sourced Bengali passages often
# have an inline citation marker like "...হয়েছে।[9] তাঁর প্রথম..." -- punctuation followed by
# "[9]", not whitespace. The old pattern required whitespace immediately after punctuation, so
# the split silently failed and merged two real sentences into one. Not id195-specific -- any
# Wikipedia-derived passage with an inline citation marker hits this.
_LOCALITY_SCAFFOLD = {"সালে","খেলেন","হয়","ছিল","করেন","হয়েছে","আছে","করে",
                       "নাম","কী","কত","যায়","হল","মুক্তি","পান"}
# NOTE (validated against real rows, e.g. id195): "প্রথম" (first) used to be excluded here to
# avoid over-triggering, but for a "her FIRST novel" question it's exactly the word that should
# anchor the match to the right (answer-bearing) sentence rather than the passage's generic
# intro sentence. Verified removing it from the exclusion list doesn't re-break the case it was
# added for -- it adds equally to both competing sentences there, so it's not decisive there.

def _split_sentences(text: str) -> list:
    parts = _SENT_SPLIT_RE.split(str(text).strip())
    return [p.strip() for p in parts if p.strip()]

def _locality_tokens(s: str) -> set:
    return {t for t in content_tokens(s) if t not in _LOCALITY_SCAFFOLD}

def _locality_tokens_ordered(s: str) -> list:
    """Order-preserving version of _locality_tokens(), for phrase-aware overlap scoring.
    BUG FOUND DURING VALIDATION (id195): tokens() returns a SET, which silently discards
    word order -- using it here would make "phrase-aware" scoring meaningless, since the
    whole point is detecting contiguous same-order runs between question and sentence.
    Must split from normalize(s) directly (a list, order preserved) instead."""
    return [t for t in (_strip_suffix(tt) for tt in normalize(s).split())
            if t not in _BN_STOPWORDS and len(t) > 1 and t not in _LOCALITY_SCAFFOLD]

def _contains_sublist(sub: list, lst: list) -> bool:
    n = len(sub)
    return any(lst[i:i + n] == sub for i in range(len(lst) - n + 1))

def _phrase_aware_overlap(q_toks: list, s_toks: list) -> int:
    """Credit each contiguous shared word-SEQUENCE once, not once per token.

    Root cause (validated against real rows, e.g. id195): plain set-overlap scoring
    double/triple-counts a multi-word proper noun (e.g. \"মহাশ্বেতা দেবী\") against a
    single distinguishing content word (e.g. \"প্রথম\") elsewhere, systematically biasing
    toward whichever sentence happens to spell out the full name -- a generic scoring
    flaw, not a fact about any one passage. This scans s_toks left to right; whenever a
    token also appears in q_toks, it greedily extends the match to the longest run that
    is still a contiguous subsequence of q_toks in the same order, and credits that
    whole run once. Operates on any repeated n-gram generically, not just this passage's
    name.
    """
    credit = 0
    i, n = 0, len(s_toks)
    while i < n:
        if s_toks[i] not in q_toks:
            i += 1
            continue
        best_len = 1
        for L in range(2, n - i + 1):
            if _contains_sublist(s_toks[i:i + L], q_toks):
                best_len = L
            else:
                break
        credit += 1
        i += best_len
    return credit

def relevant_context_for_question(ctx: str, q: str, a: str = "", min_overlap: int = 1) -> str:
    """FIX (round-6, real-run df_analysis sweep): pure question-word overlap
    gets fooled when the actual answer-bearing sentence refers to the subject
    with a pronoun/definite-article ("ক্লাবটি", "এই জেলার") instead of
    repeating it by name, while an unrelated but wordier sentence elsewhere
    repeats the proper noun many times and wins on raw overlap count.
    Confirmed on 5 real exact-match rows (id866, id496, id521, id1537,
    id1683) -- all were fed the wrong sentence, producing false num_contra=1
    / low fuzzy_contain despite the answer being verbatim-correct.
    Two-tier fix: if ANY sentence overlaps with the RESPONSE itself (the
    claimed fact, not just the general topic), restrict candidates to those
    sentences first -- a sentence containing the actual fact is a much
    stronger relevance signal than one merely repeating the subject's name.
    Question-overlap only breaks ties within that narrowed set. Falls back
    to pure question-overlap (the original behavior) when no sentence
    overlaps the response at all, or when a="" (keeps the function usable
    without a response, e.g. from any other future caller).
    """
    sents = _split_sentences(ctx)
    if len(sents) <= 1:
        return ctx
    q_toks = _locality_tokens_ordered(q)
    a_toks = _locality_tokens_ordered(a) if a else []
    if not q_toks:
        return ctx
    scored = [(_phrase_aware_overlap(q_toks, _locality_tokens_ordered(s)),
               _phrase_aware_overlap(a_toks, _locality_tokens_ordered(s)) if a_toks else 0,
               s) for s in sents]
    has_answer_overlap = any(aov > 0 for _, aov, _ in scored)
    if has_answer_overlap:
        candidates = [(qov, s) for qov, aov, s in scored if aov > 0]
    else:
        candidates = [(qov, s) for qov, aov, s in scored]
    best = max(qov for qov, _ in candidates)
    if best < min_overlap and not has_answer_overlap:
        return ctx
    return " ".join(s for qov, s in candidates if qov == best)

# superlative_hedge_mismatch(): the relative-vs-absolute-superlative cluster
# (ids 40, 98, 193) -- the question asks an absolute superlative
# (দীর্ঘতম/সর্বোচ্চ/সবচেয়ে/সর্বপ্রথম/...) and the sentence(s) actually
# relevant to the QUESTION hedge ("একটি"/"অন্যতম"/"উল্লেখযোগ্য" -- "one of,"
# "notable," "among") rather than making the absolute claim. NOTE: this
# checks the QUESTION-relevant sentence via relevant_context_for_question(),
# not the answer-matching sentence -- an earlier version anchored to the
# answer's own tokens and MISSED id40 (the hedge word "একটি" is in the
# sentence naming the glacier; the numeric answer "58km" is in the NEXT
# sentence -- two different sentences, so answer-anchoring found the wrong
# one). Re-validated on the REAL dataset rows after the fix: fires
# correctly on 2 of the 3 known failures (40, 193) and correctly does NOT
# fire on a genuine, unhedged superlative control case. Does NOT fire on
# id98 -- real-data debugging found the relevant sentence there uses the
# compound noun "পর্বতশৃঙ্গ" (mountain-peak) where the question says "শৃঙ্গ"
# (peak); these don't token-match without compound decomposition, which is
# out of scope for this pass. Flagged as a known gap, not swept under.
HEDGE_CONTEXT_RE = re.compile(r"(একটি|অন্যতম|উল্লেখযোগ্য|কয়েকটির মধ্যে|প্রধান|শীর্ষ|তালিকাভুক্ত)")
SUPERLATIVE_Q_RE = re.compile(r"(দীর্ঘতম|সর্বোচ্চ|সবচেয়ে|সর্বপ্রথম|সর্বশেষ|প্রথম|সেরা|বৃহত্তম|ক্ষুদ্রতম|উচ্চতম)")

def superlative_hedge_mismatch(q: str, ctx: str, a: str) -> int:
    if not SUPERLATIVE_Q_RE.search(q):
        return 0
    rel = relevant_context_for_question(ctx, q)
    return int(bool(HEDGE_CONTEXT_RE.search(rel)))

# HARD-RULE-GRADE version (id40 fix), deliberately separate from the LR
# feature above rather than replacing it. The LR feature's broad marker set
# (প্রধান/উল্লেখযোগ্য/শীর্ষ/তালিকাভুক্ত, matched anywhere in a possibly
# multi-sentence tied relevant-blob) is fine as one soft signal among many,
# but is NOT precise enough to gate a hard override on its own -- validated
# directly: it also fires on ids 18/98/125/145/186/216/217, several of them
# label=1 (e.g. id125 "অন্যতম প্রধান যুদ্ধ" describes the war's significance
# in the Great Game, nothing to do with the "প্রথম" ordinal in the question
# that's actually just naming which war; id145 "একটি প্রাচীন গণনা যন্ত্র" is
# the plain indefinite article "an ancient device", not a partitive hedge).
# This stricter version requires BOTH: (1) the hedge marker restricted to
# constructions that are UNAMBIGUOUSLY partitive -- অন্যতম/কয়েকটির মধ্যে
# stand alone as hedges, but একটি only counts when it directly follows a
# plural-genitive suffix (গুলোর/গুলির/দের একটি = "one of the [class]"), not
# when it's just the indefinite article before a noun; and (2) the hedge and
# a superlative word from the QUESTION's own vocabulary must appear in the
# SAME SENTENCE of the passage, not just somewhere in a tied multi-sentence
# relevant-blob. Swept the full 130-row sample with this definition: it
# fires on exactly ONE row, id40 (label=0) -- zero collateral, zero other
# matches in either direction. Safe to use as an unconditional override,
# same philosophy as the numbad rule below: the passage is directly
# contradicting the question's absolute-superlative premise in its own words.
UNAMBIGUOUS_HEDGE_RE = re.compile(r"(অন্যতম|কয়েকটির মধ্যে)")

def superlative_hedge_hardrule(q: str, ctx: str, max_dist: int = 4) -> int:
    # v6 (tightened after test_set.csv sweep -- 3 real blind spots found and
    # fixed, see below). v5 required the superlative word and the hedge
    # marker to share a sentence (via relevant_context_for_question), but
    # had NO link between them beyond that -- a sentence can easily contain
    # two unrelated claims. Swept the full 1,361-row grounded test set with
    # v5 and found exactly 3 real false-trigger cases:
    #   id432: "...অন্যতম সর্বোচ্চ পারিশ্রমিকপ্রাপ্ত ম্যানেজার..." (one of the
    #     highest-PAID managers) sits next to the question's প্রথম ("what
    #     YEAR did he first join") -- a real, well-formed hedge, just about
    #     a completely different claim (salary, not first-year-joined).
    #   id1105: প্রথম here is an EPITHET ("[Bangladesh's] first president",
    #     identifying who the person is) and অন্যতম হেজেস a totally separate
    #     claim ("one of the most influential figures") -- neither has
    #     anything to do with the birth-date question being asked.
    #   id1114 (the dangerous one): a genuinely CORRECT, unhedged answer
    #     ("Pather Panchali" really is Ray's first film) would have been
    #     wrongly flipped to hallucinated -- অন্যতম there hedges the AWARDS
    #     the film won ("...যাদের মধ্যে অন্যতম ছিল...", one of which was...),
    #     not the "first film" claim at all.
    # Fix: (1) require close TOKEN PROXIMITY (<=4 words) between the
    # superlative word and the hedge marker within the sentence, so they
    # have to belong to the same phrase, not just the same sentence; and
    # (2) require the nearby superlative word to be the SAME WORD used in
    # the question (not just any word from the category) -- this is what
    # actually kills id432, where the adjacent hedge+superlative pair
    # ("অন্যতম সর্বোচ্চ") is real but uses সর্বোচ্চ while the question uses
    # প্রথম. Partitive একটি detection also had a latent bug: Bengali fuses
    # the plural-genitive suffix onto the noun (হিমবাহ + গুলোর -> হিমবাহগুলোর,
    # ONE token), so the old exact-token-equality check against "গুলোর" as
    # a standalone token could never match real text -- fixed to an endswith
    # check on the fused token instead. Re-validated after all fixes: id40
    # still fires (দীর্ঘতম and একটি are 2 tokens apart in হিমবাহগুলোর একটি,
    # both use দীর্ঘতম), the 130-row sample sweep is unchanged (still exactly
    # one hit, id40), and all 3 test-set blind spots no longer fire. Full
    # 1,361-row grounded test-set sweep with this version: zero fires overall
    # -- so as tightened this is now a narrow correctness safety net rather
    # than something expected to move test-set metrics; kept because it's
    # zero-risk, not because it's expected to contribute volume.
    q_sup_words = set(SUPERLATIVE_Q_RE.findall(q))
    if not q_sup_words:
        return 0
    rel = relevant_context_for_question(ctx, q)
    for s in _split_sentences(rel):
        toks = normalize(s).split()
        sup_idx = [i for i, t in enumerate(toks) if any(w in t for w in q_sup_words)]
        if not sup_idx:
            continue
        hedge_idx = [i for i, t in enumerate(toks) if UNAMBIGUOUS_HEDGE_RE.search(t)]
        for i in range(len(toks) - 1):
            if toks[i].endswith(("গুলোর", "গুলির", "দের")) and toks[i + 1] == "একটি":
                hedge_idx.append(i + 1)
        for si in sup_idx:
            for hi in hedge_idx:
                if abs(si - hi) <= max_dist:
                    return 1
    return 0
def question_type(prompt: str) -> dict:
    # NOTE: only q_numeric survives downstream (needed for slot_num_ok/bad).
    # All q_* flags used to be returned as standalone model-facing features;
    # dropped per the failure-analysis round (round 3) -- every one of them
    # was noise-risk at N=130 (<10 nonzero rows) with near-random AUC. Kept
    # internal here rather than deleted outright, since slot_num_ok/bad still
    # need q_numeric's logic.
    p = prompt
    return {
        "q_numeric": int(bool(Q_NUMERIC.search(p))),
        "q_person":  int(bool(Q_PERSON.search(p))),
        "q_place":   int(bool(Q_PLACE.search(p))),
        "q_meaning": int(bool(Q_MEANING.search(p))),
        "q_math":    int(bool(Q_MATH.search(p))),
        "q_mcq":     int(bool(Q_MCQ.search(p))),
        "q_yesno":   int(bool(Q_YESNO.search(p))),
    }


# =============================================================================
# NEW (this round): six features targeting the gap categories identified via
# a manual disagreement-zone audit + a small hand-crafted contrastive probe
# set (pat23-pat36 in the augmented dataset -- kinship-relation confusion,
# hedge-removal/overconfidence, negation flip, calendar-system confusion,
# arithmetic-derivation error, evidence-selection ambiguity in long context).
# Each was confirmed as a genuine gap BEFORE writing this: the existing
# features produce IDENTICAL values for the trap/ctrl pair on these patterns
# (e.g. num_in_ctx=1 for BOTH the বঙ্গাব্দ and খ্রিস্টাব্দ year in a
# calendar-confusion row, since both numbers are literally present in the
# same sentence -- no amount of training data teaches the LR to separate two
# points with the same feature vector). All six follow the existing
# philosophy of this feature layer: return 0 (no signal) whenever the
# heuristic can't cleanly confirm something, rather than force a guess --
# same conservative posture as num_near_miss / superlative_hedge_hardrule.
# =============================================================================

# ---- TODO 3: context-hedge vs response-hedge mismatch (hedge REMOVAL) -----
# pat13/hedge_flag already catch the opposite direction (response FABRICATES
# hedging language not in context). Nothing previously compared whether the
# CONTEXT itself hedges and the response strips that hedge out, asserting
# with unearned confidence -- the more common LLM failure direction.
def context_hedge_flag(ctx_relevant: str) -> int:
    return int(bool(HEDGE_RE.search(ctx_relevant)))

# ---- TODO 2: negation-parity mismatch --------------------------------------
# Bengali negation is a verb-final particle (না/নি/নেই), which multilingual
# NLI can be inconsistent on. Cheap deterministic complement: does the
# relevant context sentence negate the claim while the response asserts it
# positively, or vice versa? Standalone-token match only (word boundaries),
# so this doesn't fire on "না" appearing as a substring inside an unrelated
# word.
# BUG FOUND during validation (pat25 handcrafted rows): "নি" is overwhelmingly a
# verb-final FUSED suffix in Bengali (রাখেনি, হয়নি), not a standalone token --
# a standalone-token-only regex silently never matched real negated verbs.
# "না"/"নেই"/"নয়" are genuinely standalone words and checked as exact tokens;
# "নি" is checked as a token-final suffix instead (length-gated to avoid a
# bare 2-char "নি" false-firing on unrelated short tokens).
def _negation_present(text: str) -> int:
    toks = normalize(text).split()
    if any(t in ("না", "নেই", "নয়") for t in toks):
        return 1
    if any(t.endswith("নি") and len(t) > 2 for t in toks):
        return 1
    return 0

def negation_parity_mismatch(ctx_relevant: str, a: str) -> int:
    return int(_negation_present(ctx_relevant) != _negation_present(a))

# ---- TODO 4: calendar-system-labeled number mismatch -----------------------
# same shape as the original relevant_context_for_question fix, but for
# calendar labels instead of sentence relevance: a বঙ্গাব্দ year and a
# খ্রিস্টাব্দ year sitting in the SAME sentence both pass num_in_ctx
# (both are literally present), even though only one is the number the
# question's calendar system actually asks for. Only fires when it can
# POSITIVELY confirm the answer's number is labeled with a DIFFERENT
# calendar system than the one asked -- never fires just because it
# "can't tell" (e.g. no calendar label present at all -> returns 0).
CALENDAR_LABEL_RE = re.compile(r"(বঙ্গাব্দ|হিজরি|খ্রিস্টাব্দ|খ্রিষ্টাব্দ)")
_CALENDAR_LABELS = ("বঙ্গাব্দ", "হিজরি", "খ্রিস্টাব্দ", "খ্রিষ্টাব্দ")

def _nearest_calendar_label(toks: list, num_idx: int, max_dist: int = 6):
    # BUG FOUND during validation (pat27 handcrafted rows): symmetric
    # same-radius windows around EACH label let two labels sitting a few
    # tokens apart in a compact sentence both claim the same number
    # ("খ্রিস্টাব্দ ১৯৮৪ সালে (বঙ্গাব্দ ১৩৯১)" -- both labels were within
    # window=4 of BOTH numbers). Nearest-label-wins can't double-claim.
    best_label, best_dist = None, None
    for i, t in enumerate(toks):
        for lab in _CALENDAR_LABELS:
            if lab in t:
                d = abs(i - num_idx)
                if d <= max_dist and (best_dist is None or d < best_dist):
                    best_dist, best_label = d, lab
    return best_label

def calendar_label_mismatch(q: str, ctx_relevant: str, a: str) -> int:
    q_cal = CALENDAR_LABEL_RE.search(q)
    if not q_cal:
        return 0
    asked_label = q_cal.group(1)
    a_nums = numbers_of(a)
    if not a_nums:
        return 0
    toks = numeralize(normalize(ctx_relevant)).split()
    for i, t in enumerate(toks):
        if not _NUM.fullmatch(t):
            continue
        val = float(t)
        if val not in a_nums:
            continue
        lbl = _nearest_calendar_label(toks, i)
        if lbl is not None and lbl != asked_label:
            return 1   # the answer's number is nearest to a DIFFERENT calendar label than asked
    return 0

# ---- TODO 5: arithmetic-derivation mismatch --------------------------------
# The highest-value gap of the six: nothing in this feature layer computes a
# derived quantity (age, duration, gap-between-two-dates) and checks it
# against the response's own number -- num_contra/num_in_ctx only check
# LITERAL presence, so a correct derived duration (e.g. 67 = 2019-1952) and
# a wrong one (57) score IDENTICALLY (both absent from context's literal
# number set). This is also, independently, the category a fine-tuned
# BanglaBERT classifier scored worst on in an earlier probe (F1=0.38) --
# encoder classifiers don't compute, so this belongs here as a symbolic
# check, not delegated to any model. Deliberately conservative: only fires
# when the question is clearly duration-shaped, the response states exactly
# one plausible small duration number, context has >=2 year-like numbers to
# derive from, and the response's number doesn't match ANY pairwise
# difference among them (±1 tolerance for inclusive/exclusive counting).
DURATION_Q_RE = re.compile(r"(কত বছর|কতদিন|কত দিন|কত মাস|বয়স|ব্যবধান|পার্থক্য)")

def arith_derivation_mismatch(q: str, ctx_relevant: str, a: str) -> int:
    if not DURATION_Q_RE.search(q):
        return 0
    a_nums = numbers_of(a)
    # BUG FOUND during validation (pat32 handcrafted rows): BN_ORDINALS parses
    # "প্রথম" (first, e.g. "first branch office") as 1.0, which collided with
    # a real duration number and broke the single-candidate gate below.
    duration_candidates = {x for x in a_nums if 1 < x <= 200}
    if len(duration_candidates) != 1:
        return 0
    resp_val = next(iter(duration_candidates))
    ctx_years = {x for x in numbers_of(ctx_relevant) if 1000 <= x <= 2035}
    if len(ctx_years) < 2:
        return 0
    valid_diffs = {round(abs(x - y)) for x in ctx_years for y in ctx_years if x != y}
    if not valid_diffs:
        return 0
    return int(not any(abs(resp_val - d) <= 1 for d in valid_diffs))

# ---- TODO 1: kinship/relation-slot binding mismatch ------------------------
# pat7/pat17-style role confusion, specialized for kinship terms: when a
# question asks for one specific relation (e.g. স্বামীর, "husband's") and
# context names TWO OR MORE relations in the same sentence (father, spouse,
# sibling...), exact_contain/fuzzy_contain can't tell which name belongs to
# which relation -- both names are literally present. No real NER here (out
# of scope for a regex layer); the proxy is token-distance: does the
# response's name sit closer, in the relevant context, to the ASKED relation
# word than to any OTHER relation word? Only fires when at least two
# distinct relation families are present AND the response's matched name
# token is unambiguously closer to a different one than the asked relation
# -- returns 0 (no signal) on any tie or when the response's name can't be
# located in context at all (can't distinguish "genuinely wrong" from
# "paraphrased/synonymous name" without over-claiming).
KIN_TERMS = {
    "বাবা":  ("বাবা", "বাবার", "পিতা", "পিতার"),
    "মা":    ("মা", "মায়ের", "মাতা", "মাতার"),
    "স্বামী": ("স্বামী", "স্বামীর"),
    "স্ত্রী": ("স্ত্রী", "স্ত্রীর"),
    "ভাই":   ("ভাই", "ভাইয়ের"),
    "বোন":   ("বোন", "বোনের"),
}

def _asked_kin_relation(q_norm: str) -> str:
    # BUG FOUND during validation (pat23 handcrafted rows): substring-in-string
    # matching lets a short relation word accidentally match INSIDE an unrelated
    # inflected token (e.g. "মা" as a bare substring of "স্বামীর" == husband's,
    # not mother's -- a Bengali-script false positive, not a logic error). Must
    # check whole question TOKENS against each variant (exact or prefix), never
    # a substring scan across the full joined string.
    q_toks = q_norm.split()
    for rel, variants in KIN_TERMS.items():
        if any(t == v or t.startswith(v) for t in q_toks for v in variants):
            return rel
    return ""

def kin_slot_mismatch(q: str, ctx_relevant: str, a: str) -> int:
    q_norm = normalize(q)
    asked_rel = _asked_kin_relation(q_norm)
    if not asked_rel:
        return 0
    _all_kin_variants = {v for variants in KIN_TERMS.values() for v in variants}
    # BUG FOUND during validation (pat23 handcrafted rows), two layers:
    # (1) a response like "স্বামীর নাম X" echoes the relation word itself
    # back -- left in, it trivially self-matches its own relation label at
    # distance 0 and always beats the real disputed name token(s).
    # (2) a response also typically echoes the SUBJECT's own name ("নাসিমা
    # আক্তারের স্বামীর নাম X") -- that echoed subject name can coincidentally
    # sit close to the WRONG relation word in a different sentence (the
    # subject is usually named once per sentence too), producing a false
    # match that has nothing to do with which name the response actually
    # claims. Both are filtered: relation-label tokens, and any token that
    # also appears among the QUESTION's own content tokens (the subject
    # re-identification), leaving only the response's actual claimed name.
    q_content_toks = {_strip_suffix(t) for t in normalize(q).split()
                       if t not in _BN_STOPWORDS and len(t) > 1}
    a_content = [t for t in (_strip_suffix(tt) for tt in normalize(a).split())
                 if t not in _BN_STOPWORDS and len(t) > 1
                 and not any(t == v or t.startswith(v) for v in _all_kin_variants)
                 and t not in q_content_toks]
    if not a_content:
        return 0
    # BUG FOUND during validation (pat23 handcrafted rows): computing distance
    # across the WHOLE relevant blob let a name at the END of one sentence
    # (grammatically bound to that sentence's relation word) score as
    # "closer" to the NEXT sentence's relation word purely because sentence
    # boundaries don't reset token distance -- e.g. "...নাম X। তাঁর স্বামী Y"
    # put X only 2 tokens from স্বামী despite belonging entirely to the
    # PRECEDING clause. Fixed by scoping distance to a single sentence at a
    # time -- a name and a relation word from different sentences never
    # compete for "closest" against each other.
    best_rel, best_dist = None, None
    for sent in _split_sentences(ctx_relevant):
        toks = normalize(sent).split()
        rel_positions = {}
        for rel, variants in KIN_TERMS.items():
            pos = [i for i, t in enumerate(toks) if any(t == v or t.startswith(v) for v in variants)]
            if pos:
                rel_positions[rel] = pos
        if not rel_positions:
            continue
        for tok in a_content:
            idxs = [i for i, t in enumerate(toks) if t == tok or t.startswith(tok)]
            for i in idxs:
                for rel, positions in rel_positions.items():
                    d = min(abs(i - p) for p in positions)
                    if best_dist is None or d < best_dist:
                        best_dist, best_rel = d, rel
    if best_rel is None:
        return 0   # response's name not found in any single relevant sentence -- don't guess
    # need at least two competing relation families ANYWHERE in the relevant
    # blob for this to even be a slot-confusion risk, not just a one-relation mention
    all_rels_seen = {rel for sent in _split_sentences(ctx_relevant)
                      for rel, variants in KIN_TERMS.items()
                      if any(t == v or t.startswith(v) for t in normalize(sent).split() for v in variants)}
    if len(all_rels_seen) < 2:
        return 0
    return int(best_rel != asked_rel)

# ---- TODO 7: evidence-selection ambiguity (tie count) ----------------------
# cot_ctx_truncated/support_ctx_truncated only fire on raw length exceeding
# the judge's prompt budget. A separate risk exists even under budget: a
# long, multi-topic passage where several sentences tie on question-overlap
# score, making relevant_context_for_question's selection genuinely
# ambiguous rather than truncated. Reuses the exact same
# _phrase_aware_overlap/_locality_tokens_ordered machinery already validated
# for sentence-relevance scoring, just counts ties instead of picking one.
def relevant_sentence_tie_count(ctx: str, q: str) -> int:
    sents = _split_sentences(ctx)
    if len(sents) <= 1:
        return 0
    q_toks = _locality_tokens_ordered(q)
    if not q_toks:
        return 0
    scores = [_phrase_aware_overlap(q_toks, _locality_tokens_ordered(s)) for s in sents]
    best = max(scores)
    if best == 0:
        return 0
    return sum(1 for sc in scores if sc == best) - 1


def det_features(row) -> dict:
    ctx, q, a = row["context"], row["prompt_bn"], row["response_bn"]
    qt = question_type(q)   # internal only -- NOT merged into f (all q_* dropped as model features)
    f = {}
    na, nq = normalize(a), normalize(q)
    a_toks = tokens(a)
    a_nums = numbers_of(a)
    f["resp_len"]      = len(na)
    f["resp_n_tokens"] = len(a_toks)
    f["resp_has_num"]  = int(bool(a_nums))
    f["resp_abstain"]  = int(bool(ABSTAIN.search(a)))
    f["resp_yes"]      = int(bool(BN_YES.search(a.strip())))
    f["resp_no"]       = int(bool(BN_NO.search(a.strip())))
    # answer-type vs question-type agreement (wrong-slot signal)
    f["slot_num_ok"]   = int(qt["q_numeric"] and bool(a_nums))
    f["slot_num_bad"]  = int(qt["q_numeric"] and not a_nums and f["resp_n_tokens"] <= 6 and not f["resp_abstain"])

    # ---- NEW candidate features: answer/question-only (no context needed) ----
    f["resp_q_ratio"]     = f["resp_len"] / max(len(nq), 1)
    f["hedge_flag"]       = int(bool(HEDGE_RE.search(a)))
    f["multi_answer"]     = int(bool(MULTI_ANS_RE.search(a.strip())))
    f["year_implausible"] = int(f["resp_has_num"] and not _year_plausible(a_nums))

    # Phase A is grounded-only (scoped at load time), so `ctx` is always
    # provided here -- the old closed-book zero-fill branch never fired and
    # has been removed.
    c_toks, c_nums, nc = tokens(ctx), numbers_of(ctx), normalize(ctx)
    cov = len(a_toks & c_toks) / len(a_toks) if a_toks else 0.0
    f["cov_tokens"]     = cov   # deliberately WHOLE-PASSAGE -- this feeds is_silent
                                # regime routing (section 7b), a broad topical-relevance
                                # check, not a fact-verification check. Scoping it down
                                # to the relevant sentence would change silent-row
                                # routing, a different mechanism than the one being
                                # fixed here -- left untouched, out of scope for this fix.
    # number-form normalization for containment specifically -- a
    # spelled-out Bengali number ("আঠারশ বত্রিশ") should string-match the
    # same value in digit form in the context ("১৮৩২") and vice versa.
    # numeralize() unifies both to digits before the containment checks;
    # na/nc themselves stay as-is for every other feature (resp_len etc.)
    # NEW: restrict containment to the sentence(s) most relevant to the
    # QUESTION, not the whole passage -- see relevant_context_for_question()
    # above for what this does and doesn't fix (validated on real rows).
    nc_relevant = normalize(relevant_context_for_question(ctx, q, a))
    na_num, nc_num = numeralize(na), numeralize(nc_relevant)
    f["fuzzy_contain"]  = fuzzy_partial(na_num, nc_num) / 100.0
    f["exact_contain"]  = int(na_num in nc_num and len(na_num) >= 2)

    # ---- FIX: num_in_ctx/num_contra/num_near_miss/content_cov/char3_contain
    # were checking against numbers_of(ctx)/content_tokens(ctx) -- the FULL
    # passage -- which lets a wrong value "pass" purely because it happens to
    # appear SOMEWHERE ELSE in a long passage for an unrelated fact (a birth
    # year borrowed from an award year mentioned three sentences later, a
    # sentence's minimum-value quoted when the maximum was asked, etc).
    # Confirmed as the dominant real-world LR failure mode via df_analysis.csv
    # audit: 8 of 27 error rows on a live run shared this exact signature
    # (num_in_ctx=1 despite the specific claimed fact being wrong). Reusing
    # the SAME relevant-sentence scope exact_contain/fuzzy_contain already
    # use above, rather than inventing a second selection mechanism.
    c_nums_relevant = numbers_of(nc_relevant)
    f["num_in_ctx"]     = int(bool(a_nums) and a_nums <= c_nums_relevant)
    f["num_contra"]     = int(bool(a_nums) and not (a_nums <= c_nums_relevant))
    # near-miss: answer number close to a context number but not equal (I1)
    near = 0
    for x in a_nums - c_nums_relevant:
        if any(0 < abs(x - y) <= max(2.0, 0.02 * max(abs(x), abs(y))) for y in c_nums_relevant):
            near = 1
    f["num_near_miss"]  = near
    f["ctx_len"]        = len(nc)

    # ---- NEW candidate features: context-dependent ----
    ca_toks = content_tokens(a)
    cc_toks_relevant = content_tokens(nc_relevant)   # FIX: same relevant-sentence scope as above,
                                                      # was content_tokens(ctx) (whole passage)
    f["content_cov"]    = (len(ca_toks & cc_toks_relevant) / len(ca_toks)) if ca_toks else 0.0
    f["char3_contain"]  = char_ngram_containment(na, nc_relevant)   # FIX: was char_ngram_containment(na, nc)
    f["resp_ctx_ratio"] = f["resp_len"] / max(len(nc), 1)           # unchanged -- length ratio, not containment
    f["num_extra_ratio"] = (len(a_nums - c_nums_relevant) / len(a_nums)) if a_nums else 0.0

    # ---- NEW: context-truncation detection, tied to the ACTUAL prompt
    # budgets judge_worker.py uses (not an arbitrary threshold). NLI is
    # NOT subject to either of these -- run_nli() chunks + max-pools the
    # FULL context regardless of length -- so a row flagged here has a
    # judge signal built on incomplete input while its NLI/det signals
    # remain built on the complete context. len(ctx) (raw chars, not the
    # normalized nc) matches what the worker actually slices on.
    raw_ctx_len = len(str(ctx))
    cot_budget     = CFG.COT_MAX_CTX_CHARS               # judge_cot's prompt budget
    support_budget = CFG.MAX_CTX_TOKENS * 3              # judge_support's prompt budget (char approx)
    f["cot_ctx_truncated"]      = int(raw_ctx_len > cot_budget)
    f["support_ctx_truncated"]  = int(raw_ctx_len > support_budget)
    f["cot_ctx_overflow_chars"] = max(0, raw_ctx_len - cot_budget)

    # NEW: relative-vs-absolute-superlative mismatch (see helper above).
    f["superlative_hedge_mismatch"] = superlative_hedge_mismatch(q, ctx, a)  # a unused now, kept for signature stability
    # HARD-RULE-ONLY signal -- deliberately NOT fed to the LR (see note
    # above on why it's kept separate from superlative_hedge_mismatch).
    # Consumed directly by apply_hard_rules() via det, not via X_train.
    f["superlative_hedge_hardrule"] = superlative_hedge_hardrule(q, ctx)

    # ---- NEW (this round): kinship/hedge-removal/negation/calendar/arithmetic/
    # evidence-ambiguity features -- see helper block above det_features() for
    # the full rationale on each. All computed against nc_relevant (the same
    # relevant-sentence scope exact_contain/num_in_ctx already use) except
    # relevant_sentence_tie_count, which needs the FULL context to count ties
    # across all sentences, not just the winning one.
    # kin_slot_mismatch (TODO 1) is DELIBERATELY NOT included here. After
    # three rounds of bug-fixing (sentence-boundary distance leakage,
    # relation-word self-echo, subject-name echo) it settled at firing on
    # essentially nothing (1/896 rows on the full augmented set, and 0/2 on
    # its own handcrafted validation pairs) -- pure token-distance proximity
    # is not a reliable enough proxy for grammatical relation-binding without
    # real NER/dependency parsing. Left defined below (not deleted) as a
    # documented starting point for a future pass, not wired into `f`.
    f["context_hedge_flag"]         = context_hedge_flag(nc_relevant)
    # SAFETY GATE (added after reviewing real LR-coefficient + disagreement
    # data from the first run): hedge_removed had one confirmed false
    # positive (id1659, "মহীনের ঘোড়াগুলি" first-rock-band row) where
    # relevant_context_for_question's answer-priority tier locked onto an
    # unrelated LATER sentence (about musical genre, not the founding claim)
    # purely because it happened to repeat the response's own text -- and
    # that unrelated sentence coincidentally contained a HEDGE_RE word. Gated
    # on `not exact_contain`: when the response is already a full, literal
    # match somewhere in the relevant context, that is much stronger direct
    # evidence of faithfulness than a coincidentally-picked-up hedge word
    # should be allowed to override. Verified this doesn't cost the feature
    # its real positives: genuine hedge-removal cases (pat24) score
    # exact_contain=0 already, since a confidently-rewritten answer is a
    # paraphrase, not a verbatim substring -- so this gate is free there and
    # only suppresses the coincidental-sentence failure mode.
    f["hedge_removed"]              = int(f["context_hedge_flag"] and not f["hedge_flag"] and not f["exact_contain"])
    f["negation_parity_mismatch"]   = negation_parity_mismatch(nc_relevant, a)
    f["calendar_label_mismatch"]    = calendar_label_mismatch(q, nc_relevant, a)
    f["arith_derivation_mismatch"]  = arith_derivation_mismatch(q, nc_relevant, a)
    f["relevant_sentence_ties"]     = relevant_sentence_tie_count(ctx, q)
    return f

def build_det(df: pd.DataFrame) -> pd.DataFrame:
    feats = pd.DataFrame([det_features(r) for _, r in df.iterrows()], index=df.index)
    return feats

train_det = build_det(train_df)
print("deterministic features:", list(train_det.columns))

## 4 · Neural scorer A — multilingual NLI
mDeBERTa-v3-base-xnli, chunked max-entailment/max-contradiction pooling, plus the `nli_neutral` "silent-row" signal. Identical to the inference notebook's NLI stage, run once on `train_df`.

In [ ]:
def resolve_local_model_dir(name_hint: str) -> str:
    """Find a local model directory under CFG.LOCAL_MODEL_DIRS (the attached
    Kaggle "models" dataset) whose path contains name_hint. There is NO Hugging
    Face id fallback appended here -- if nothing matches, this raises
    FileNotFoundError, which is meant to surface as a hard runtime error instead
    of silently degrading to a different/smaller model.
    """
    hint = name_hint.split("/")[-1].lower().replace("_", "-")
    hits = []
    for root in CFG.LOCAL_MODEL_DIRS:
        if not os.path.isdir(root):
            continue
        for p in glob.glob(os.path.join(root, "**/config.json"), recursive=True):
            d = os.path.dirname(p)
            if hint in d.lower().replace("_", "-"):
                hits.append(d)
    if not hits:
        raise FileNotFoundError(
            f"No local model directory found matching '{name_hint}' under "
            f"{CFG.LOCAL_MODEL_DIRS} -- attach the corresponding Kaggle model "
            f"dataset. There is no Hugging Face fallback for this model."
        )
    hits.sort(key=lambda p: p.count(os.sep))   # least-nested match wins
    return hits[0]

nli_scores_train = None
_nli_tok = _nli_mdl = None    # loaded once, shared by both splits

def _load_nli():
    """Load the single local NLI model. No candidate loop and no fallback: a
    load failure propagates as a real exception (hard runtime error), by design.
    """
    global _nli_tok, _nli_mdl
    if _nli_mdl is not None:
        return
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    path = resolve_local_model_dir(CFG.NLI_MODEL_NAME)
    _nli_tok = AutoTokenizer.from_pretrained(path)
    _nli_mdl = AutoModelForSequenceClassification.from_pretrained(
        path, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
    ).to(DEVICE).eval()
    print("NLI model:", path)
    ARCH["nli_model"] = path

def run_nli(df: pd.DataFrame) -> pd.DataFrame:
    """nli_entail / nli_contra: whether the PROVIDED context entails/contradicts
    the (question, answer) pair, max-pooled over context chunks. NaN wherever a
    row produced no chunks at all (shouldn't happen for grounded rows, but kept
    as a safety net)."""
    _load_nli()
    tok, mdl = _nli_tok, _nli_mdl

    id2label = {int(k): v.lower() for k, v in mdl.config.id2label.items()}
    ent_id = [i for i, v in id2label.items() if "entail" in v][0]
    con_id = [i for i, v in id2label.items() if "contra" in v][0]

    def chunks(text, size=380, overlap=120):
        text = text.strip()
        if len(text) <= size:
            return [text]
        out, i = [], 0
        while i < len(text):
            out.append(text[i:i + size]); i += size - overlap
        return out

    cols = {c: pd.Series(np.nan, index=df.index, dtype=float)
            for c in ("nli_entail", "nli_contra")}

    pairs, owner = [], []
    for i in df.index:                                            # provided context (always present -- grounded-only)
        r = df.loc[i]
        hyp = f"প্রশ্ন: {r['prompt_bn']} উত্তর: {r['response_bn']}"
        for ch in chunks(r["context"]):
            pairs.append((ch, hyp)); owner.append(i)

    probs = np.zeros((len(pairs), mdl.config.num_labels), dtype=np.float32)
    with torch.no_grad():
        for s in range(0, len(pairs), CFG.BATCH_NLI):
            if not budget_ok():
                print("runtime budget hit inside NLI — truncating"); break
            batch = pairs[s:s + CFG.BATCH_NLI]
            enc = tok([p for p, _ in batch], [h for _, h in batch],
                      truncation=True, max_length=512, padding=True, return_tensors="pt").to(DEVICE)
            probs[s:s + len(batch)] = torch.softmax(mdl(**enc).logits, dim=-1).float().cpu().numpy()

    agg = defaultdict(float)
    for k, i in enumerate(owner):
        agg[(i, "nli_entail")] = max(agg[(i, "nli_entail")], float(probs[k, ent_id]))
        agg[(i, "nli_contra")] = max(agg[(i, "nli_contra")], float(probs[k, con_id]))
    for (i, cname), v in agg.items():
        cols[cname][i] = v

    # ---- NEW: nli_neutral -- the "silent" signal. Rather than max-pool a
    # third (neutral) class across chunks (ambiguous which chunk should win),
    # derive it from the two aggregates already computed: how much
    # probability mass is unaccounted for by the STRONGEST entailment
    # evidence and the STRONGEST contradiction evidence found ANYWHERE in
    # the context. High nli_neutral == no chunk anywhere strongly supports
    # OR contradicts the claim -- exactly the "requires reasoning the NLI
    # model can't do, not lexical matching" signature this is meant to catch.
    cols["nli_neutral"] = (1.0 - cols["nli_entail"].fillna(0) - cols["nli_contra"].fillna(0)).clip(0, 1)
    # keep NaN where the row never got a nli_entail/contra pair at all (no
    # context chunks were produced for it) rather than falsely reporting 1.0
    cols["nli_neutral"][cols["nli_entail"].isna()] = np.nan

    return pd.DataFrame(cols)

def _free_nli():
    global _nli_mdl, _nli_tok
    _nli_mdl = _nli_tok = None
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

if CFG.USE_NLI and TORCH_OK:
    try:
        t = time.time()
        nli_scores_train = run_nli(train_df)   # model loaded once,
        print(f"NLI stage done in {(time.time()-t)/60:.1f} min")
    except Exception as e:
        note_fallback("NLI", f"stage failed ({type(e).__name__}: {e}) -> NLI features dropped; "
                             f"grounded rows rely on deterministic features + LLM judge")
    finally:
        _free_nli()
elif not TORCH_OK:
    note_fallback("NLI", "torch unavailable -> NLI disabled")
else:
    print("NLI disabled by config (CFG.USE_NLI=False)")

## 5 · GPU eviction before the judge stage
Verbatim from the inference notebook — frees the NLI model's VRAM before vLLM loads the judge (no co-residency invariant, same as the inference notebook).

In [ ]:
import gc as _gc
try:
    _free_nli()
except Exception:
    pass

for _nm in ("_nli_mdl", "_nli_tok", "_bge", "Qt", "Xg", "_rag_Q"):
    if _nm in globals():
        try: globals()[_nm] = None
        except Exception: pass

# multiple GC passes: one collect() often misses reference cycles holding
# CUDA tensors (autograd graphs / hooks) -- matters a lot with only ~14GB total.
for _ in range(3):
    _gc.collect()

if TORCH_OK and torch.cuda.is_available():
    for _d in range(torch.cuda.device_count()):
        try:
            with torch.cuda.device(_d):
                torch.cuda.synchronize()
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
                torch.cuda.reset_peak_memory_stats()
        except Exception:
            pass
    free_gb  = [round(torch.cuda.mem_get_info(d)[0] / 1e9, 2) for d in range(torch.cuda.device_count())]
    total_gb = [round(torch.cuda.mem_get_info(d)[1] / 1e9, 2) for d in range(torch.cuda.device_count())]
    print(f"pre-judge GPU eviction done | free/total GiB per device: {list(zip(free_gb, total_gb))}")
    for d, (f, t) in enumerate(zip(free_gb, total_gb)):
        if f < 0.5 * t:
            note_fallback("gpu/memory",
                f"GPU {d} only has {f:.1f}/{t:.1f} GiB free before the judge stage -> "
                f"an earlier stage likely leaked GPU memory; judge init may need to "
                f"fall back to a smaller model or lower gpu_memory_utilization.")
else:
    print("pre-judge GPU eviction done (CPU mode)")

## 6 · Neural scorer B — Qwen3-14B-AWQ judge (vLLM subprocess)
Verbatim vLLM subprocess bootstrap from the inference notebook (TP=2 across both T4s, prefix caching, continuous batching, AWQ forced to the plain kernel to avoid the Marlin/T4 misdetection issue, eager mode to avoid the CUDA-graph hang point). No sample/test references exist in this cell — it's pure infra — so it is reused completely unmodified.

In [ ]:
# =============================================================================
# §5.1 · vLLM judge — fresh `python3` subprocess bootstrap (Phase-B style)
# -----------------------------------------------------------------------------
# The judge now runs on this notebook's own pinned stack (installed once, at
# the very top, before any heavy import). There is no venv, no ABI isolation,
# and no HF device_map fallback: this cell writes judge_worker.py to disk as
# plaintext, sanity-checks it via a fresh `python3 -c "import vllm, torch..."`
# subprocess (the system interpreter reading the pinned packages off disk --
# exactly how Phase B's cell 90 launches run_half.py), and py_compiles the
# worker. If either check fails, the failure is recorded (fallback + arch
# logs flushed) and the notebook HARD-FAILS via RuntimeError -- there is no
# fallback judge, so we fail fast here rather than burning an hour of NLI
# before discovering the judge can't run.
# =============================================================================
import os, subprocess, sys, py_compile
from pathlib import Path

# ---------------------------------------------------------------------------
# NCCL / tensor-parallel hang mitigation (2xT4 in a sandboxed container).
# Symptom seen: vLLM engine allocates GPU memory on BOTH GPUs (weights loaded
# fine) but then sits at 0% GPU compute / 0% CPU indefinitely -- a deadlock,
# not slow progress. Root cause: vLLM's tensor_parallel_size=2 workers can
# hang on an NCCL peer-to-peer synchronization barrier in container/cloud
# environments that don't fully expose GPU P2P/NVLink the way NCCL's default
# transport expects. Forcing NCCL onto a safer transport (no P2P, no shared
# memory) before the subprocess is spawned -- these env vars are inherited by
# the judge subprocess launched from run_judge_vllm_subprocess() below --
# avoids that hang path. NCCL_DEBUG=WARN (not INFO) surfaces real transport
# errors without flooding the log with per-call trace spam.
# ---------------------------------------------------------------------------
os.environ.setdefault("NCCL_P2P_DISABLE", "1")
os.environ.setdefault("NCCL_SHM_DISABLE", "1")
os.environ.setdefault("NCCL_DEBUG", "WARN")
print("NCCL hang-mitigation env set: NCCL_P2P_DISABLE=1 NCCL_SHM_DISABLE=1 NCCL_DEBUG=WARN")

JUDGE_PY      = "python3"                              # system interpreter -- same launch style as cell 90
WORKER_SCRIPT = "/kaggle/working/judge_worker.py"

_WORKER_SRC = r'''#!/usr/bin/env python
# =============================================================================
# judge_worker.py — runs on the notebook's own pinned stack (vllm 0.6.0 /
# torch 2.4.0 / transformers 4.45.2 / numpy 2.0.0) as a FRESH `python3`
# subprocess launched by the kernel (see notebook cell 17). No venv, no ABI
# isolation needed: the kernel installed this exact stack, on disk, before
# importing anything, so this subprocess and the kernel agree on every pin.
#
# Prompts / few-shot exemplars / verdict parsing are copied VERBATIM from the
# original HF judge so the signals are identical -- this is a speed change
# (vLLM TP=2 + prefix caching + continuous batching), not a signal change.
# Only the plumbing (argv, parquet in/out, single model load) is new.
#
# IN : parquet with columns id, prompt_bn, response_bn, context
# OUT: parquet with columns id, judge_support, judge_cot,
#      judge_cot_text, cot_prompt_tokens, cot_over_budget, support_prompt_tokens,
#      support_over_budget
# =============================================================================
import os, re, sys, json, gc, argparse, traceback
import numpy as np
import pandas as pd
import torch

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("VLLM_ATTENTION_BACKEND", "XFORMERS")       # T4: no FlashAttention2
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")

def eprint(*a):
    print(*a, file=sys.stderr, flush=True)
    print(*a, flush=True)

try:
    from vllm import LLM, SamplingParams
except Exception:
    eprint("FATAL: could not import vllm inside the venv subprocess:")
    eprint(traceback.format_exc())
    sys.exit(2)

# --------------------------------------------------------------------- prompts
SYS_GROUNDED = ("তুমি একজন কঠোর তথ্য-যাচাইকারী। শুধুমাত্র প্রদত্ত অনুচ্ছেদের ভিত্তিতে বিচার করবে; "
                "বাইরের জ্ঞান ব্যবহার করবে না। উত্তরটি অনুচ্ছেদ দ্বারা সমর্থিত হলে 'হ্যাঁ', "
                "না হলে বা অনুচ্ছেদের সাথে সাংঘর্ষিক হলে 'না' — শুধু এই একটি শব্দ লিখবে। "
                "সংখ্যা অঙ্কে বা কথায় লেখা থাকলে একই মান হলে সমর্থিত ধরবে। "
                "উত্তরের শব্দ/সংখ্যা অনুচ্ছেদের যেকোনো জায়গায় উপস্থিত থাকলেই তা সমর্থিত ধরবে না — "
                "সেটি অবশ্যই এই নির্দিষ্ট প্রশ্নের সঠিক উত্তর হিসেবে অনুচ্ছেদে বলা থাকতে হবে, "
                "অনুচ্ছেদের অন্য কোনো ব্যক্তি/স্থান/তারিখ সম্পর্কিত হলে চলবে না। "
                "অনুচ্ছেদে আংশিক বা আপেক্ষিক তথ্য (যেমন 'উল্লেখিতদের মধ্যে সবচেয়ে বড়', 'সর্বশেষ উল্লেখিত') "
                "থাকলে সেটিকে প্রশ্নে জিজ্ঞাসিত সম্পূর্ণ/চূড়ান্ত দাবির (যেমন 'সবচেয়ে বড়', 'সর্বশেষ ব্যক্তি') "
                "সমর্থন হিসেবে ধরবে না, যদি না অনুচ্ছেদ স্পষ্টভাবে সেই সম্পূর্ণ দাবিটি নিশ্চিত করে। "
                "গাণিতিক উত্তর হলে নিজে হিসাব করে যাচাই করবে। বহুনির্বাচনী প্রশ্নে প্রস্তাবিত উত্তরটি "
                "সঠিক অপশনই কিনা যাচাই করবে, শুধু বিশ্বাসযোগ্য শোনালেই যথেষ্ট নয়। "
                "প্রথমে নিজে অনুচ্ছেদ থেকে প্রশ্নের সঠিক উত্তরটি বের করার চেষ্টা করবে, তারপর প্রস্তাবিত "
                "উত্তরের সাথে মেলাবে। একই সঠিক উত্তর একাধিকভাবে সঠিকভাবে প্রকাশ করা যেতে পারে (পূর্ণ নাম "
                "বনাম সংক্ষিপ্ত রূপ, ভিন্ন কিন্তু সমতুল্য বাক্যাংশ, বা প্রশ্নে জিজ্ঞাসিত তথ্যটিই যে নির্দিষ্ট "
                "বিবরণ) — তুমি যা সঠিক উত্তর বলে নির্ধারণ করেছ তার সাথে বাস্তবিকভাবে মিললে তা গ্রহণযোগ্য। "
                "কিন্তু প্রস্তাবিত উত্তরটি প্রকৃতপক্ষে সেই সঠিক উত্তরের সাথে না মিললে — তা প্রাসঙ্গিক, সত্য, "
                "বা অনুচ্ছেদে উপস্থিত কিছুর নাম হলেও — 'না' (অসমর্থিত) বলবে। "
                "অসম্পূর্ণতা হ্যালুসিনেশন নয় — উত্তরে বলা প্রতিটি তথ্য অনুচ্ছেদ দ্বারা সমর্থিত ও সঠিক হলে, "
                "এবং প্রশ্নের সাথে প্রাসঙ্গিক অতিরিক্ত কোনো তথ্য বাদ পড়লেও, উত্তরটি 'হ্যাঁ' (সমর্থিত) ধরবে — "
                "শুধুমাত্র উত্তরে ভুল বা মনগড়া তথ্য থাকলেই 'না' বলবে। ব্যতিক্রম: বাদ পড়া অংশটি যদি এমন হয় "
                "যে তা বাদ দেওয়ার ফলে পাঠক ভুল ধারণা পাবেন (যেমন একাধিক সঠিক উত্তরের একটিমাত্র উল্লেখ করে "
                "সেটিকেই একমাত্র/সম্পূর্ণ উত্তর বলে উপস্থাপন করা), তখন সেটি সমর্থিত ধরবে না।")
def grounded_user(r, max_ctx_tokens):
    ctx = str(r["context"])[:max_ctx_tokens * 3]
    return (f"অনুচ্ছেদ:\n{ctx}\n\nপ্রশ্ন: {r['prompt_bn']}\n"
            f"প্রস্তাবিত উত্তর: {r['response_bn']}\n\n"
            f"প্রস্তাবিত উত্তরটি কি অনুচ্ছেদ দ্বারা সমর্থিত?")

COT_FEWSHOT = [
    ("উইন্ডোজে ইউনিকোড ভিত্তিক বাংলা লেখার জন্য মেহদী হাসান খান ২০০৩ সালে অভ্র কীবোর্ড তৈরির কাজ শুরু করেন।",
     "অভ্র কিবোর্ড কে উদ্ভাবন করেন ?", "মেহদী হাসান খান",
     "The context explicitly credits Mehdi Hasan Khan with creating Avro. The response matches.", 1),
    ("পশ্চিমের মালভূমির সর্বোচ্চ অংশ অযোধ্যা পাহাড় (৬৭০ মিটার)।",
     "পুরুলিয়া জেলার অযোধ্যা পাহাড়ের উচ্চতা কত ?", "প্রায় ৭২০ মিটার।",
     "The context states 670 metres, but the response says 720 metres. The number contradicts the context.", 0),
    ("শহরের তিনটি প্রধান প্রকাশনা সংস্থার মধ্যে বর্ণমালা প্রকাশনী সবচেয়ে পুরনো, যা ১৯৫৮ সালে প্রতিষ্ঠিত হয়।",
     "দেশের সবচেয়ে পুরনো প্রকাশনা সংস্থা কোনটি?", "বর্ণমালা প্রকাশনী",
     "The context only says Barnamala is the oldest AMONG THE THREE PUBLISHERS MENTIONED here, "
     "not the oldest in the whole country -- the passage's limited scope doesn't support the "
     "absolute claim the question is asking about.", 0),
    ("১৯৯০ সাল পর্যন্ত এই ইউনিয়নের চেয়ারম্যান ছিলেন করিম উদ্দিন, যিনি টানা তিন মেয়াদে দায়িত্ব পালন করেন।",
     "এই ইউনিয়নের সর্বশেষ চেয়ারম্যান কে ছিলেন?", "করিম উদ্দিন",
     "The context only states Karim Uddin was chairman UP TO 1990 -- it says nothing about who came "
     "after, so this confirms he was the last one MENTIONED in this passage, not the actual last "
     "chairman.", 0),
    ("রহিম উদ্দিন ১৯৮৫ সালে জাতীয় গণতান্ত্রিক ফ্রন্ট (জাগফ্রন্ট) প্রতিষ্ঠা করেন এবং আজীবন এর সভাপতি ছিলেন।",
     "রহিম উদ্দিন কোন রাজনৈতিক দলের সভাপতি ছিলেন?", "জাগফ্রন্ট",
     "The context founds Rahim Uddin's party as জাতীয় গণতান্ত্রিক ফ্রন্ট and gives its "
     "abbreviation in parentheses as জাগফ্রন্ট -- same organization, just abbreviated. Matches.", 1),
    ("করিম হোসেন ১৯৯০ সালে বাংলাদেশ প্রগতিশীল আন্দোলন প্রতিষ্ঠা করেন। তার প্রতিদ্বন্দ্বী দল ছিল "
     "জাতীয় গণতান্ত্রিক ফ্রন্ট (জাগফ্রন্ট)।",
     "করিম হোসেন কোন দল প্রতিষ্ঠা করেন?", "জাগফ্রন্ট",
     "Karim Hossain founded বাংলাদেশ প্রগতিশীল আন্দোলন -- জাগফ্রন্ট is his RIVAL party mentioned "
     "in the same passage, a different entity. The response names the wrong organization even "
     "though জাগফ্রন্ট genuinely appears (as someone else's abbreviation) in the passage.", 0),
    ("প্রতিষ্ঠানটি ১৯৭৬ সালে স্টিভ জবস, ওয়াকার ফিন্‌চ ও রোনাল্ড হেইস -- এই তিনজন মিলে প্রতিষ্ঠা করেন।",
     "প্রতিষ্ঠানটির প্রতিষ্ঠাতা কারা?", "ওয়াকার ফিন্‌চ ও রোনাল্ড হেইস",
     "The response names two of the three co-founders the context states (Walker Finch and "
     "Ronald Hayes) and doesn't claim they are the only ones or state anything false -- naming "
     "a true subset of a correct multi-part answer is incomplete, not fabricated, so this is "
     "faithful.", 1),
    ("প্রতিযোগিতাটি মূলত ভারত, শ্রীলঙ্কা ও বাংলাদেশ জুড়ে বিভিন্ন ভেন্যুতে অনুষ্ঠিত হয়েছিল। "
     "ফাইনাল ম্যাচটি হয়েছিল মুম্বাইয়ের একটি স্টেডিয়ামে।",
     "প্রতিযোগিতাটি কোথায় অনুষ্ঠিত হয়েছিল?", "মুম্বাইয়ে",
     "The context states the tournament as a whole was held across India, Sri Lanka, and "
     "Bangladesh -- Mumbai only hosted the final. Presenting the final's single venue as the "
     "answer to where the WHOLE tournament was held is misleading by omission (it implies a "
     "multi-country event was a single-city one), which crosses into hallucination even though "
     "Mumbai genuinely did host the final.", 0),
    ("চলচ্চিত্রটি বিখ্যাত বিজ্ঞানী মেরি অ্যাডামসের জীবনী অবলম্বনে নির্মিত এবং এতে অভিনেত্রী লরা বেনেট "
     "মূল ভূমিকায় অভিনয় করেন।",
     "চলচ্চিত্রে লরা বেনেট অভিনীত চরিত্রটির নাম কী ছিল?", "মেরি অ্যাডামস",
     "The context doesn't literally write \"Laura Bennett played Mary Adams\", but it states the "
     "film is a biopic of Mary Adams and that Laura Bennett plays the lead role -- in a biopic, "
     "the lead role IS the subject. This is the obvious structural inference the context "
     "supports, so the response is faithful.", 1),
    ("রফিক হাসান ২০০৫ সালে সাবিনা ইয়াসমিনকে বিবাহ করেন। তাঁদের দুই সন্তান রয়েছে।",
     "সাবিনা ইয়াসমিনের স্বামীর নাম কী?", "রফিক হাসান",
     "The context states \"Rafiq Hasan married Sabina Yasmin\" -- this relationship holds in "
     "both directions, so it equally answers \"who is Sabina Yasmin's husband\" even though the "
     "context is phrased from Rafiq Hasan's side. Faithful.", 1),
    ("উদ্ভিদকোষে খাদ্য সঞ্চয়কারী মূল অঙ্গাণু হলো ক্লোরোপ্লাস্ট, যা মেসোফিল কোষে অবস্থিত।",
     "উদ্ভিদকোষে খাদ্য সঞ্চয়ের প্রধান স্থান কোনটি?", "মেসোফিল কলা",
     "The context says the food-storing organelle (chloroplast) is located in mesophyll CELLS "
     "(কোষ). The response says মেসোফিল কলা (mesophyll TISSUE) -- কলা is a standard, valid Bengali "
     "biology term for tissue, and cells/tissue here refer to the same underlying location; this "
     "is terminology-level equivalence, not a factual error. Faithful.", 1),
     # Specificity: vague hedge in place of a stated fact
    ("বাংলাদেশ ১৯৭১ সালের ১৬ ডিসেম্বর বিজয় অর্জন করে। মোট নয় মাস স্থায়ী মুক্তিযুদ্ধে প্রায় ৩০ লক্ষ মানুষ প্রাণ হারান।",
     "মুক্তিযুদ্ধ কত মাস স্থায়ী ছিল?", "বেশ কয়েক মাস",
     "The context explicitly states the war lasted nine months (নয় মাস). The response gives only a "
     "vague hedge ('several months') instead of the specific figure the context clearly provides. "
     "A vague answer in place of an available specific fact is a specificity failure, not a safe "
     "partial answer.", 0),
    # Comprehension: true content, but answers a different question than asked
    ("ঢাকা বাংলাদেশের রাজধানী এবং জনসংখ্যার দিক থেকে দেশের বৃহত্তম শহর। এটি বুড়িগঙ্গা নদীর তীরে অবস্থিত।",
     "ঢাকা কোন নদীর তীরে অবস্থিত?", "ঢাকা বাংলাদেশের রাজধানী।",
     "Both the response and the context statement it draws from are true, but the response answers "
     "'what is Dhaka' instead of the question actually asked -- which river Dhaka sits on. Being "
     "factually true is not enough if the response doesn't address the specific thing asked; this "
     "is a comprehension failure, not a faithful answer.", 0),
]


SYS_COT = (
    "You are a meticulous Bengali fact-checking expert. Given a PROMPT and a RESPONSE (and "
    "and a CONTEXT), decide whether the RESPONSE is faithful/correct.\n"
    "- If CONTEXT is present, the response must be fully supported by it (numbers, names, dates).\n"
    "- If CONTEXT is present but provides no use as in you have to calculate a math equation or tackle a grammar quesion, use accurate world knowledge; catch wrong dates, names, attributions, and "
    "arithmetic errors.These cases are rare so USE THIS AS LAST RESORT ONLY. MOST QUESTIONS WILL HAVE THE ANSWERS IN THE CONTEXT\n"
    "- First work out for yourself what the correct answer to the question actually is (from the "
    "context). The same correct answer can be "
    "validly expressed in more than one way (a full name vs. an abbreviation, an equivalent "
    "phrasing, or the specific detail that is itself the fact being asked about) -- accept the "
    "response if it genuinely matches what you determined the correct answer to be, in any "
    "reasonable valid form. But if the response does not actually match the correct answer -- even "
    "if it names something true, related, or present in the passage -- declare it hallucinated.\n"
    "- For arithmetic/numeric questions with no relevant context(AGAIN THESE ARE RARE), compute the answer yourself step by step BEFORE judging "
    "the response -- do not just pattern-match the presence of numbers.\n"
    "- For multiple-choice questions, verify the response is the OBJECTIVELY CORRECT option, not "
    "merely a plausible-sounding one.\n"
    "- For word-meaning/idiom questions (অর্থ, ভাবার্থ, বাগধারা), judge SEMANTIC equivalence to the "
    "true meaning, not surface lexical overlap with the context or question.\n"
    "- Do NOT equate a claim that is merely relative or partial within the passage (e.g. 'the "
    "oldest of the ones mentioned here', 'the last one mentioned in this passage', 'the highest "
    "among the listed items') with an ABSOLUTE superlative asked about in the question (e.g. 'the "
    "oldest overall', 'the actual last', 'the highest in the whole region/country'). The passage's "
    "coverage may be incomplete -- if it doesn't explicitly make the exact absolute claim the "
    "question asks about, the response is NOT supported, even if it's consistent with everything "
    "the passage happens to mention.\n"
    "- Incompleteness is NOT hallucination. If every fact the response states is true and "
    "supported, but it simply leaves out other true, relevant information (e.g. naming two of "
    "three co-founders, or omitting a detail the question didn't specifically ask about), that "
    "is faithful (VERDICT: 1) -- do not penalize an answer for not being exhaustive. Only actual "
    "fabrication, contradiction, or wrong attribution is hallucination. EXCEPTION: if what's left "
    "out makes the response misleading -- e.g. presenting one of several correct answers as if it "
    "were the only/complete one, or a narrow fact as if it answered a broader question -- that "
    "IS hallucination, because the omission itself creates a false impression.\n"
    "- ANSWERS AT THE WRONG LEVEL OF SPECIFICITY ARE HALLUCINATION. A vague hedge ('কিছু', "
    "'অনেক', 'জানা নেই', 'বিভিন্ন কারণে') offered in place of a specific fact the context clearly "
    "states is NOT a safe partial answer -- it's a failure to state the fact, so declare it "
    "hallucinated (VERDICT: 0). Likewise, an answer padded with extra specific detail the context "
    "does not support (inventing a sub-category, a precise figure, or a named example the context "
    "never gives) is hallucination even if the general direction is right.\n"
    "- CHECK THAT THE RESPONSE ANSWERS THE QUESTION ACTUALLY ASKED, not a nearby or related "
    "question. A response can be 100% true and 100% present in the context and still be "
    "hallucinated if it answers a different question than the one asked (e.g. describing what a "
    "country is instead of naming its capital). If the response's content doesn't address the "
    "specific thing the question asks for, declare it hallucinated (VERDICT: 0), even though "
    "nothing in it is factually false on its own.\n"
    "- Script or language of the response (Bengali script, Roman-script Bangla, or English words "
    "mixed in) is NEVER by itself a reason to mark something hallucinated. Judge only the factual "
    "content once you understand what it says, regardless of which script it's written in.\n"
    "- Make the OBVIOUS structural inference the context supports, even if it isn't spelled out "
    "verbatim. If the context states a biographical film's SUBJECT and the question asks who an "
    "actor played, the subject IS the character. If the context states 'A married B', that also "
    "answers 'who is B's spouse' (relationships work in both directions) -- don't require the "
    "reverse phrasing to appear literally in the text.\n"
    "- Recognize valid terminology equivalence, including a Bengali term/translation for an "
    "English/scientific term the context uses (or vice versa) -- e.g. context saying 'mesophyll "
    "cells' supports a response saying 'মেসোফিল কলা' (the standard Bengali translation); do not "
    "require an exact string match of technical vocabulary across languages.\n"
    "Reason briefly in 1-3 sentences, then end with exactly one line: 'VERDICT: 1' (faithful) or "
    "'VERDICT: 0' (hallucinated)."
)
COT_FINAL_Q = "Reason briefly, then output 'VERDICT: 1' or 'VERDICT: 0'."

def _cot_user(ctx, prompt, resp, cot_max_ctx_chars):
    parts = []
    if ctx:
        parts.append(f"CONTEXT:\n{str(ctx)[:cot_max_ctx_chars]}")
    parts += [f"PROMPT:\n{prompt}", f"RESPONSE:\n{resp}", COT_FINAL_Q]
    return "\n\n".join(parts)

def _cot_context_for(row):
    return str(row.get("context", "") or "")

def _cot_messages(row, n_fewshot, cot_max_ctx_chars):
    m = [{"role": "system", "content": SYS_COT}]
    for c, p, r, reason, v in COT_FEWSHOT[:n_fewshot]:
        m.append({"role": "user", "content": _cot_user(c, p, r, cot_max_ctx_chars)})
        m.append({"role": "assistant", "content": f"{reason}\nVERDICT: {v}"})
    m.append({"role": "user", "content": _cot_user(_cot_context_for(row), row["prompt_bn"],
                                                     row["response_bn"], cot_max_ctx_chars)})
    return m

_VERDICT_RE = re.compile(r"VERDICT\s*[:：]?\s*([01])")
def _parse_verdict(text):
    hits = _VERDICT_RE.findall(text or "")
    if hits:
        return int(hits[-1])
    tail = (text or "").strip()[-40:]
    d = re.findall(r"[01]", tail)
    return int(d[-1]) if d else None


def _apply_chat_noThink(tok, msgs):
    """Phase A judge always runs with Qwen3 thinking OFF -- wraps
    apply_chat_template with a TypeError fallback for tokenizers that don't
    accept enable_thinking (defensive; the Phase A judge model is Qwen3)."""
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                       enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


class VJudge:
    P_YES_TOPK = 20  # must match/exceed the topk requested in p_yes(); see __init__
    def __init__(self, model, max_len, gpu_util, n_gpu, quant, eager, dtype, seed):
        kw = dict(model=model, tokenizer=model, tensor_parallel_size=n_gpu,
                  dtype=dtype, gpu_memory_utilization=gpu_util,
                  max_model_len=max_len, enable_prefix_caching=True,
                  enforce_eager=eager, trust_remote_code=True, seed=seed,
                  # p_yes requests completion-level top-k logprobs (see p_yes
                  # docstring); vLLM rejects any SamplingParams(logprobs=N)
                  # request above this ceiling with an immediate ValueError
                  # (no GPU work attempted at all -- exactly what was seen:
                  # every p_yes block failed instantly with a bare ValueError,
                  # no CUDA/OOM traces). Raise the ceiling to match what
                  # p_yes actually requests (P_YES_TOPK below).
                  max_logprobs=VJudge.P_YES_TOPK)
        if quant:
            kw["quantization"] = quant
        # ---- somadhan-style build-with-retry: a bad quant/CUDA-graph combo on
        # Turing (T4, sm_75) can throw on first construction (e.g. Marlin kernel
        # misdetection, graph-capture OOM). Retry once, forced to eager mode with
        # quantization auto-detected, before giving up -- same pattern proven in
        # somadhan_pipeline's worker.py build_engine().
        try:
            self.llm = LLM(**kw)
        except Exception as e:
            eprint(f"[worker] engine init failed ({type(e).__name__}: {e}); "
                   f"retrying eager + auto-detect quant")
            kw["enforce_eager"] = True
            kw.pop("quantization", None)
            self.llm = LLM(**kw)
        self.tok = self.llm.get_tokenizer()
        if self.tok.pad_token_id is None:
            self.tok.pad_token = self.tok.eos_token
        self.max_len = max_len

    def _chat(self, system, user):
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        # Phase A judge: thinking always OFF (Qwen3 native reasoning is reserved
        # for Phase B's B2 math/derivable/escalation worker only, per design).
        try:
            return self.tok.apply_chat_template(msgs, tokenize=False,
                                                 add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def p_yes(self, prompts, block, options=("হ্যাঁ", "না"), topk=None):
        """MEMORY-EFFICIENT REDESIGN (root-cause fix, not a workaround).

        OLD approach: teacher-force [prompt + option] and request
        prompt_logprobs=1 over the WHOLE sequence. vLLM's prompt_logprobs
        forces a full-vocabulary softmax at EVERY prompt position (not just
        the last one it actually needs for generation) -- so cost scales with
        CONTEXT LENGTH. On ~1000-token contexts x 100+ concurrent sequences
        this blew past any GPU memory budget: confirmed root cause of a real
        OOM cascade (CoT's plain generate() never showed this, since ordinary
        decoding only needs the LAST position's logits -- same reason this
        redesign is safe).

        NEW approach: generate exactly ONE token per prompt (no option
        appended) with completion-level `logprobs=topk` (NOT prompt_logprobs).
        This is a single last-position softmax per sequence, INDEPENDENT of
        context length -- identical cost profile to the CoT stage, which
        never OOM'd. vLLM always includes the actually-sampled (top-1) token's
        logprob in the returned dict regardless of `topk`, plus up to
        `topk-1` runners-up. Our system prompts explicitly instruct the model
        to answer with ONLY one of these two short Bengali words, so the
        correct option is essentially always top-1 or very near it; the rare
        case where NEITHER option appears in the top-`topk` degrades
        gracefully to a neutral 0.5 score instead of crashing anything.

        Semantics are unchanged: returns p(yes) in [0,1] via the same
        sigmoid-of-logit-difference the rest of the pipeline already expects.
        """
        if not prompts:
            return np.zeros(0, dtype=np.float32)
        topk = self.P_YES_TOPK if topk is None else min(topk, self.P_YES_TOPK)
        opt_first_id = [self.tok(o, add_special_tokens=False)["input_ids"][0] for o in options]
        cap = self.max_len - 8
        seqs = [self.tok(p, add_special_tokens=False)["input_ids"][-cap:] for p in prompts]
        out = np.full((len(prompts), len(options)), -20.0, dtype=np.float32)
        sp = SamplingParams(temperature=0.0, max_tokens=1, logprobs=topk)
        cur_block = max(1, block)
        s = 0
        consecutive_hard_fail = 0
        while s < len(seqs):
            end = min(s + cur_block, len(seqs))
            bs = seqs[s:end]
            try:
                ros = self.llm.generate([{"prompt_token_ids": s} for s in bs],
                                        sampling_params=sp, use_tqdm=False)
            except torch.cuda.OutOfMemoryError:
                gc.collect(); torch.cuda.empty_cache()
                if cur_block == 1:
                    eprint(f"[worker]   CUDA OOM on a single sequence during p_yes "
                           f"(offset {s}) -> neutral score assigned")
                    s = end
                    continue
                cur_block = max(1, cur_block // 2)
                eprint(f"[worker]   CUDA OOM during p_yes -> reduced block size to "
                       f"{cur_block} and retrying offset {s} (kept for later chunks too)")
                continue   # retry the SAME offset with the smaller cur_block
            except Exception as e:
                consecutive_hard_fail += 1
                eprint(f"[worker]   p_yes block failed non-recoverably at offset {s}: "
                       f"{type(e).__name__}: {e} -> neutral scores for this block "
                       f"(consecutive hard failures: {consecutive_hard_fail})")
                gc.collect(); torch.cuda.empty_cache()
                if consecutive_hard_fail >= 5 and cur_block == 1:
                    eprint(f"[worker]   {consecutive_hard_fail} consecutive non-recoverable engine "
                           f"errors at block size 1 -> assuming the vLLM engine is corrupted; "
                           f"bulk-assigning neutral scores to the remaining "
                           f"{len(seqs) - s} sequences and aborting this p_yes call (this "
                           f"circuit-breaker exists so a corrupted engine can't burn 15-20 "
                           f"minutes retrying doomed single-item calls one at a time)")
                    s = len(seqs)
                    continue
                s = end
                continue
            consecutive_hard_fail = 0
            for pi, ro in zip(range(s, end), ros):
                comp = ro.outputs[0]
                lp_dict = {}
                if comp.logprobs and len(comp.logprobs) > 0 and comp.logprobs[0]:
                    lp_dict = {tid: obj.logprob for tid, obj in comp.logprobs[0].items()}
                for oi, tid in enumerate(opt_first_id):
                    out[pi, oi] = lp_dict.get(tid, -20.0)
            s = end
        return 1.0 / (1.0 + np.exp(out[:, 1] - out[:, 0]))

    def generate(self, prompts, max_new, do_sample, temperature, seed, _oom_salt=0):
        if not prompts:
            return []
        use_seed = (int(seed) + _oom_salt) if (do_sample and seed is not None) else None
        sp = SamplingParams(temperature=(float(temperature) if do_sample else 0.0),
                            top_p=(0.9 if do_sample else 1.0), max_tokens=int(max_new),
                            seed=use_seed)
        try:
            ros = self.llm.generate(prompts, sp, use_tqdm=False)
            return [o.outputs[0].text.strip() for o in ros]
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            if len(prompts) == 1:
                eprint("[worker]   CUDA OOM generating a single prompt -> returned empty string")
                return [""]
            mid = len(prompts) // 2
            eprint(f"[worker]   CUDA OOM generating a batch of {len(prompts)} -> retrying as two "
                   f"batches of {mid} and {len(prompts)-mid}")
            # distinct salts so a sampled retry doesn't reuse the identical seed
            # (and therefore identical draw) across both halves
            return (self.generate(prompts[:mid], max_new, do_sample, temperature, seed, _oom_salt + 1)
                    + self.generate(prompts[mid:], max_new, do_sample, temperature, seed, _oom_salt + 2))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--in", dest="inp", required=True)
    ap.add_argument("--out", dest="out", required=True)
    ap.add_argument("--model", required=True)
    ap.add_argument("--max-len", type=int, required=True)
    ap.add_argument("--gpu-util", type=float, required=True)
    ap.add_argument("--n-gpu", type=int, required=True)
    ap.add_argument("--dtype", default="float16")
    ap.add_argument("--quant", default="")
    ap.add_argument("--eager", action="store_true")
    ap.add_argument("--seed", type=int, required=True)
    ap.add_argument("--cot-fewshot", type=int, required=True)
    ap.add_argument("--cot-max-ctx", type=int, required=True)
    ap.add_argument("--cot-max-new", type=int, required=True)
    ap.add_argument("--cot-block", type=int, required=True)
    ap.add_argument("--max-ctx-tokens", type=int, required=True)
    ap.add_argument("--logprob-block", type=int, required=True)
    a = ap.parse_args()

    # BUGFIX safety net: --cot-fewshot has silently fallen out of sync with
    # COT_FEWSHOT three separate times as exemplars were appended in later
    # rounds (round-3 superlative/hedge, round-4 arithmetic/MCQ, abbreviation-
    # equivalence) -- COT_FEWSHOT[:n_fewshot] just silently drops anything past
    # the cap instead of erroring, so none of those rounds' prompt fixes were
    # ever actually shown to the judge in any real run. Fail loud instead.
    assert a.cot_fewshot == len(COT_FEWSHOT), (
        f"--cot-fewshot={a.cot_fewshot} != len(COT_FEWSHOT)={len(COT_FEWSHOT)} -- "
        f"someone added/removed exemplars without updating CFG.COT_N_FEWSHOT, which "
        f"would silently truncate the few-shot list again. Fix CFG.COT_N_FEWSHOT."
    )

    eprint(f"[worker] loading input: {a.inp}")
    df = pd.read_parquet(a.inp).reset_index(drop=True)
    if "context" not in df.columns:
        df["context"] = ""
    df["context"] = df["context"].fillna("")
    eprint(f"[worker] {len(df)} rows (grounded-only -- Phase A never sends closed-book rows here)")

    eprint(f"[worker] initializing vLLM: model={a.model} n_gpu={a.n_gpu} "
           f"max_len={a.max_len} gpu_util={a.gpu_util} dtype={a.dtype} eager={a.eager}")
    j = VJudge(a.model, a.max_len, a.gpu_util, a.n_gpu, (a.quant or None), a.eager, a.dtype, a.seed)
    eprint("[worker] vLLM engine ready")

    def _clear(label):
        """Aggressive GPU cache clearing between stages, as requested --
        cheap insurance even though the p_yes redesign above should make
        this far less necessary than it was before."""
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        eprint(f"[worker]   (GPU cache cleared after {label})")

    n = len(df)
    judge_support = np.full(n, np.nan, dtype=np.float32)
    judge_cot     = np.full(n, np.nan, dtype=np.float32)
    judge_cot_text = [None] * n   # NEW: the FULL raw greedy CoT rationale text --
                                   # free (no extra generation), just persisting what's
                                   # already produced. Supersedes the length-only capture:
                                   # length is derivable from this, and the full text is
                                   # what actually lets you audit *why* the judge got a
                                   # row wrong (qwen_full_log_sample.json downstream).
    # NEW: real token-budget accounting (replaces the char-length proxy computed
    # in det_features). Measured once per row below, using the ACTUAL tokenizer
    # on the FULL untruncated prompt -- these are ground truth, not an estimate.
    cot_prompt_tokens     = np.zeros(n, dtype=np.int32)
    cot_over_budget       = np.zeros(n, dtype=bool)
    support_prompt_tokens = np.zeros(n, dtype=np.int32)
    support_over_budget   = np.zeros(n, dtype=bool)

    # ---- NEW: real CoT token-budget check, using the FULL untruncated context
    # (10**9 == no char slicing) run through the ACTUAL tokenizer, compared
    # against the model's real usable window (max_model_len minus the
    # reserved generation budget). This is ground truth, unlike the
    # det_features char-length proxy -- a row can look "fine" by chars but
    # still overflow once actually tokenized (Bengali is not 1 char/token).
    eprint("[worker] stage: CoT token-budget accounting")
    cot_token_budget = j.max_len - a.cot_max_new
    for i in df.index:
        full_text = _apply_chat_noThink(j.tok, _cot_messages(df.loc[i], a.cot_fewshot, 10**9))
        ntok = len(j.tok.encode(full_text))
        cot_prompt_tokens[i] = ntok
        cot_over_budget[i] = ntok > cot_token_budget
    eprint(f"[worker]   {int(cot_over_budget.sum())}/{n} rows exceed the real CoT window "
           f"({cot_token_budget} tokens, budget = max_len {j.max_len} - cot_max_new {a.cot_max_new})")

    # ---- CoT verdict on every row (+ self-consistency on closed-book) --------
    eprint("[worker] stage: CoT verdicts")
    # Bengali is not 1 char/token, so a char-based cot_max_ctx cap can still
    # blow the real (token) budget on a handful of token-dense rows -- the
    # budget-accounting stage above measures this correctly, but was
    # diagnostic-only; this loop now actually enforces it per row before any
    # prompt reaches vLLM. Without this, ONE over-length prompt raises
    # ValueError inside llm.generate(), kills this worker, and (per the
    # dispatcher above) aborts the whole run -- there is no fallback judge.
    prompts = []
    for i in df.index:
        ctx_cap = a.cot_max_ctx
        text = _apply_chat_noThink(j.tok, _cot_messages(df.loc[i], a.cot_fewshot, ctx_cap))
        ntok = len(j.tok.encode(text))
        tries = 0
        while ntok > cot_token_budget and ctx_cap > 200 and tries < 6:
            ctx_cap = int(ctx_cap * 0.7)
            text = _apply_chat_noThink(j.tok, _cot_messages(df.loc[i], a.cot_fewshot, ctx_cap))
            ntok = len(j.tok.encode(text))
            tries += 1
        if ntok > cot_token_budget:
            # last resort: hard-truncate the tokenized prompt itself, keeping the
            # tail (chat-template close + the actual question typically sits at
            # the end of _cot_messages' rendering, so the tail is what matters).
            ids = j.tok.encode(text)[-cot_token_budget:]
            text = j.tok.decode(ids)
        prompts.append(text)
    votes = {i: [] for i in df.index}
    for s in range(0, len(prompts), a.cot_block):
        chunk = prompts[s:s + a.cot_block]
        idx_chunk = df.index[s:s + a.cot_block]
        greedy = j.generate(chunk, a.cot_max_new, False, 0.0, None)
        for i, txt in zip(idx_chunk, greedy):
            v = _parse_verdict(txt)
            if v is not None:
                votes[i].append(v)
            judge_cot_text[i] = (txt or "").strip()   # NEW
        eprint(f"[worker]   CoT greedy {min(s+a.cot_block, len(prompts))}/{len(prompts)}")
    for i in df.index:
        if votes[i]:
            judge_cot[i] = float(np.mean(votes[i]))
    _clear("CoT stage")

    # ---- grounded option-logprob verdict (the only branch Phase A ever hits --
    # every row here is grounded; the old closed/rag/self-answer branches were
    # removed, they never fired) ---------------------------------------------
    # ---- NEW: real support (p_yes) token-budget check, same approach as
    # the CoT one above -- full untruncated context, actual tokenizer,
    # real usable window. p_yes only needs to generate ~1 token, so the
    # reserved budget is tiny compared to CoT's.
    eprint(f"[worker] stage: support token-budget accounting ({n} rows)")
    support_token_budget = j.max_len - 8
    for i in df.index:
        full_text = j._chat(SYS_GROUNDED, grounded_user(df.loc[i], 10**9))
        ntok = len(j.tok.encode(full_text))
        support_prompt_tokens[i] = ntok
        support_over_budget[i] = ntok > support_token_budget
    eprint(f"[worker]   {int(support_over_budget.sum())}/{n} grounded rows exceed "
           f"the real support window ({support_token_budget} tokens)")

    eprint(f"[worker] stage: grounded p_yes ({n} rows)")
    # Same class of bug as the CoT stage above: grounded_user()'s truncation is
    # a char-based heuristic (max_ctx_tokens * 3 chars), and support_over_budget
    # was measured but never enforced. Enforce it here per row before p_yes ever
    # sees the prompt -- one over-length request crashes this worker (and with
    # it the whole run; no fallback judge exists).
    support_prompts = []
    for i in df.index:
        ctx_cap = a.max_ctx_tokens
        text = j._chat(SYS_GROUNDED, grounded_user(df.loc[i], ctx_cap))
        ntok = len(j.tok.encode(text))
        tries = 0
        while ntok > support_token_budget and ctx_cap > 100 and tries < 6:
            ctx_cap = int(ctx_cap * 0.7)
            text = j._chat(SYS_GROUNDED, grounded_user(df.loc[i], ctx_cap))
            ntok = len(j.tok.encode(text))
            tries += 1
        if ntok > support_token_budget:
            ids = j.tok.encode(text)[-support_token_budget:]
            text = j.tok.decode(ids)
        support_prompts.append(text)
    p = j.p_yes(support_prompts, a.logprob_block)
    for k, i in enumerate(df.index):
        judge_support[i] = p[k]
    _clear("grounded p_yes")

    out = pd.DataFrame({
        "id": df["id"].values,
        "judge_support": judge_support, "judge_cot": judge_cot,
        "judge_cot_text": judge_cot_text,   # NEW (full raw text, not just length)
        "cot_prompt_tokens": cot_prompt_tokens,         # NEW
        "cot_over_budget": cot_over_budget,             # NEW
        "support_prompt_tokens": support_prompt_tokens, # NEW
        "support_over_budget": support_over_budget,     # NEW
    })
    out.to_parquet(a.out, index=False)
    eprint(f"[worker] DONE -> wrote {len(out)} rows to {a.out}")


if __name__ == "__main__":
    try:
        main()
    except Exception:
        eprint("[worker] FATAL EXCEPTION:")
        eprint(traceback.format_exc())
        sys.exit(1)
'''

Path(WORKER_SCRIPT).write_text(_WORKER_SRC)
print(f"judge_worker.py written -> {WORKER_SCRIPT} ({len(_WORKER_SRC)} bytes)")

# ---- sanity check: fresh subprocess import, then py_compile -----------------
VLLM_SUBPROCESS_OK = False
r = subprocess.run(
    [JUDGE_PY, "-c", "import vllm, torch; print(\'SANITY_OK\', vllm.__version__, torch.__version__, torch.version.cuda)"],
    capture_output=True, text=True, timeout=180,
)
print("---- subprocess stdout ----")
print(r.stdout.strip())
print("---- subprocess stderr ----")
print(r.stderr.strip()[-3000:])
print("---- end subprocess output ----")
_sanity_ok = (r.returncode == 0) and ("SANITY_OK" in r.stdout)

_compile_ok = False
if _sanity_ok:
    try:
        py_compile.compile(WORKER_SCRIPT, doraise=True)
        _compile_ok = True
        print("py_compile OK: judge_worker.py has no syntax errors")
    except py_compile.PyCompileError as e:
        print(f"py_compile FAILED: {e}")

VLLM_SUBPROCESS_OK = _sanity_ok and _compile_ok

if not VLLM_SUBPROCESS_OK:
    _reason = ("subprocess sanity check failed" if not _sanity_ok else "judge_worker.py failed py_compile")
    note_fallback("judge/backend",
        f"vLLM subprocess judge bootstrap FAILED ({_reason}, exit {r.returncode}). "
        f"Exact stderr tail above. There is NO fallback judge -- aborting the run now "
        f"rather than burning an hour of NLI before discovering the judge can't run.")
    flush_arch_log()
    raise RuntimeError(f"vLLM subprocess judge bootstrap failed: {_reason} (exit {r.returncode})")

print("=" * 78)
print(f"VLLM_SUBPROCESS_OK = {VLLM_SUBPROCESS_OK}")
print("JudgeBackend : vLLM via fresh python3 subprocess (ONLY judge backend) — TP=2, fp16, "
      f"XFORMERS, enforce_eager={CFG.VLLM_ENFORCE_EAGER}, gpu_util={CFG.VLLM_GPU_UTIL} "
      "(both are cell-3 defaults; see rationale comments there)")
print("=" * 78)


### Judge dispatch — `judge_support` + `judge_cot`, run once on `train_df`
Every judge interaction is still logged to `qwen_responses_train.json` for offline error analysis, exactly as the inference notebook logs to `qwen_responses.json`.

In [ ]:
import subprocess

# --------------------------- Qwen response log ------------------------------
# Every judge interaction — option-scored verdicts AND free-form self-answers —
# is recorded and flushed to OUTPUT_DIR/qwen_responses.json (CFG.QWEN_LOG_FILE).
# The vLLM subprocess path logs little into it today; behavior kept identical.
QWEN_LOG = []
JUDGE_COT_TEXT_STORE = {}   # split -> pd.Series(id-indexed full CoT text)

def _row_key(df, i):
    return (df.loc[i, "id"] if "id" in df.columns else int(i))

def log_qwen(stage, split, row_id, prompt, response=None, p_yes=None, extra=None):
    rec = {"stage": stage, "split": split, "row_id": row_id,
           "prompt_head": str(prompt)[:600], "elapsed_h": round(elapsed_h(), 3)}
    if response is not None:
        rec["response"] = str(response)
    if p_yes is not None:
        rec["p_yes"] = round(float(p_yes), 6)
    if extra:
        rec.update(extra)
    QWEN_LOG.append(rec)

def flush_qwen_log():
    path = os.path.join(OUTPUT_DIR, CFG.QWEN_LOG_FILE)
    try:
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"judge_model": ARCH.get("judge_model"),
                       "judge_quant": ARCH.get("judge_quant"),
                       "n_records": len(QWEN_LOG), "records": QWEN_LOG},
                      f, ensure_ascii=False, indent=1)
        print(f"qwen response log written: {path}  ({len(QWEN_LOG)} records)")
    except Exception as e:
        print("could not write qwen response log:", e)

def _cuda_cleanup():
    """Release cached allocator blocks — call between batches/stages."""
    gc.collect()
    if TORCH_OK and torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


# ---------------------- architecture fallback log --------------------------
def flush_arch_log():
    """Persist the full architecture + every fallback across the whole pipeline."""
    path = os.path.join(OUTPUT_DIR, CFG.ARCH_LOG_FILE)
    payload = {
        "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
        "elapsed_h": round(elapsed_h(), 3),
        "architecture": ARCH,
        "n_fallbacks": len(FALLBACK_LOG),
        "fallbacks": FALLBACK_LOG,
    }
    try:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=1)
        print(f"architecture/fallback log written: {path}  ({len(FALLBACK_LOG)} fallbacks)")
    except Exception as e:
        print("could not write architecture log:", e)


# =============================================================================
# vLLM judge via a fresh `python3` subprocess (the ONLY judge backend — no
# HF device_map fallback exists). Reuses JUDGE_PY + WORKER_SCRIPT prepared in
# cell 17. Writes rows to parquet, launches the subprocess on judge_worker.py,
# reads results back into a DataFrame keyed by id. Prints the
# worker's full stdout/stderr for debugging either way.
# =============================================================================

# subprocess.Popen inherits the FULL parent environment by default, including
# PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True (set in cell 3, needed for
# the deterministic/ensemble in-kernel torch, if any). vLLM's TP=2 custom
# all-reduce shares CUDA tensors between its two worker processes via
# `.untyped_storage()._share_cuda_()`, and PyTorch's expandable-segments
# allocator explicitly forbids sharing tensors it allocated across processes
# -- confirmed root cause of "RuntimeError: Tensors allocated with
# expandable_segments:True cannot be shared between processes" during
# init_worker_distributed_environment. This is a plumbing fix only (env var
# for the subprocess's own allocator, not a judge-signal change): give the
# judge subprocess its own copy of the environment with expandable_segments
# turned off, while still inheriting NCCL_*/VLLM_ATTENTION_BACKEND/etc. set
# in the kernel (cell 17).
_JUDGE_SUBPROCESS_ENV = os.environ.copy()
_JUDGE_SUBPROCESS_ENV["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:False"

def run_judge_vllm_subprocess(df: pd.DataFrame, split: str) -> pd.DataFrame:
    work_in  = os.path.join(OUTPUT_DIR, f"judge_in_{split}.parquet")
    work_out = os.path.join(OUTPUT_DIR, f"judge_out_{split}.parquet")

    payload = pd.DataFrame({
        "id":              df["id"].astype(str) if "id" in df.columns else df.index.astype(str),
        "prompt_bn":       df["prompt_bn"],
        "response_bn":     df["response_bn"],
        "context":         df.get("context", pd.Series("", index=df.index)).fillna(""),
    })
    payload.to_parquet(work_in, index=False)

    model = resolve_local_model_dir(CFG.JUDGE_MODEL_NAME)   # local only, no candidate list
    n_gpu = max(1, torch.cuda.device_count()) if (TORCH_OK and torch.cuda.is_available()) else 1

    cmd = [JUDGE_PY, WORKER_SCRIPT,
           "--in", work_in, "--out", work_out, "--model", model,
           "--max-len", str(CFG.VLLM_MAX_MODEL_LEN),
           "--gpu-util", str(CFG.VLLM_GPU_UTIL),
           "--n-gpu", str(n_gpu),
           "--dtype", CFG.VLLM_DTYPE,
           "--seed", str(SEED),
           "--cot-fewshot", str(CFG.COT_N_FEWSHOT),
           "--cot-max-ctx", str(CFG.COT_MAX_CTX_CHARS),
           "--cot-max-new", str(CFG.COT_MAX_NEW_TOKENS),
           "--cot-block", str(CFG.COT_VLLM_BLOCK),
           "--max-ctx-tokens", str(CFG.MAX_CTX_TOKENS),
           "--logprob-block", str(CFG.VLLM_LOGPROB_BLOCK)]
    if CFG.VLLM_QUANTIZATION:
        cmd += ["--quant", CFG.VLLM_QUANTIZATION]
    if CFG.VLLM_ENFORCE_EAGER:
        cmd.append("--eager")

    print(f"[vLLM subprocess] launching judge worker [{split}]: model={model} n_gpu={n_gpu}")
    print(f"[vLLM subprocess] command: {' '.join(cmd)}")
    print(f"[vLLM subprocess] PYTORCH_CUDA_ALLOC_CONF override for this subprocess: "
          f"{_JUDGE_SUBPROCESS_ENV['PYTORCH_CUDA_ALLOC_CONF']} (kernel's own setting is "
          f"left untouched -- this only affects the child process's allocator)")
    print(f"---- worker live output [{split}] (streamed as it prints; no buffering) ----")
    t0 = time.time()
    # Popen + line-by-line read instead of subprocess.run(capture_output=True):
    # capture_output BUFFERS everything until the process exits, so a long
    # judge run (minutes to hours) shows NOTHING in the notebook the whole
    # time even while the GPUs are genuinely at 100% -- exactly what was seen
    # here. Streaming stdout/stderr (merged) as it's produced gives real-time
    # progress (the worker's own "[worker] stage: ..." / "CoT greedy X/N"
    # lines) instead of a long silent gap followed by one giant dump.
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=_JUDGE_SUBPROCESS_ENV)
    tail_lines = []
    for line in proc.stdout:
        line = line.rstrip("\n")
        print(f"[worker:{split}] {line}")
        tail_lines.append(line)
        if len(tail_lines) > 400:
            tail_lines = tail_lines[-400:]
    proc.wait()
    print(f"---- end worker live output [{split}] (exit {proc.returncode}) ----")

    if proc.returncode != 0 or not os.path.exists(work_out):
        raise RuntimeError(
            f"vLLM subprocess judge FAILED for split={split} (exit {proc.returncode}). "
            f"Last output lines:\n" + "\n".join(tail_lines[-40:])
        )

    print(f"[vLLM subprocess] judge [{split}] finished in {(time.time()-t0)/60:.1f} min")
    res = pd.read_parquet(work_out)
    res["id"] = res["id"].astype(str)
    res = res.set_index("id").reindex(payload["id"].values)
    res.index = df.index  # realign to df's original index

    # NEW candidate features derived from EXISTING judge outputs -- zero extra
    # inference cost, just recombinations of judge_support/judge_cot:
    #   judge_margin   -- |p(yes) - 0.5| * 2, in [0,1]: how CONFIDENT the p_yes
    #                      scorer was, independent of which direction it leaned.
    #   judge_disagree -- 1 if judge_cot and judge_support land on opposite
    #                      sides of 0.5.
    #   judge_cot_len  -- character length of the CoT rationale.
    js = res["judge_support"].astype(float)
    jc = res["judge_cot"].astype(float)
    judge_cot_text_series = res["judge_cot_text"] if "judge_cot_text" in res.columns else pd.Series([None] * len(df), index=df.index)
    judge_cot_len = judge_cot_text_series.apply(lambda t: float(len(t)) if isinstance(t, str) else np.nan)
    JUDGE_COT_TEXT_STORE[split] = judge_cot_text_series   # full raw text, kept OUT of the
                                                           # numeric feature matrix (assemble() only
                                                           # takes numeric columns).

    # ---- real token-budget masking (replaces the char-length regime split
    # from before). cot_over_budget/support_over_budget come from the
    # worker's ACTUAL tokenizer measurement on the full untruncated prompt.
    # Every judge-derived feature built from a truncated prompt gets set to
    # NaN for that specific row, rather than swapping the WHOLE row to a
    # different model -- a row that only lost judge_cot still keeps a
    # perfectly good judge_support, and vice versa.
    cot_over_budget     = res["cot_over_budget"].astype(bool) if "cot_over_budget" in res.columns else pd.Series(False, index=df.index)
    support_over_budget = res["support_over_budget"].astype(bool) if "support_over_budget" in res.columns else pd.Series(False, index=df.index)
    cot_prompt_tokens     = res["cot_prompt_tokens"].astype(float) if "cot_prompt_tokens" in res.columns else pd.Series(np.nan, index=df.index)
    support_prompt_tokens = res["support_prompt_tokens"].astype(float) if "support_prompt_tokens" in res.columns else pd.Series(np.nan, index=df.index)

    jc_masked = jc.copy();                 jc_masked[cot_over_budget] = np.nan
    judge_cot_len_masked = judge_cot_len.copy();  judge_cot_len_masked[cot_over_budget] = np.nan
    js_masked = js.copy();                 js_masked[support_over_budget] = np.nan

    judge_margin = (js_masked - 0.5).abs() * 2   # NaN propagates automatically when js_masked is NaN
    judge_disagree = ((js_masked >= 0.5) != (jc_masked >= 0.5)).astype(float)
    judge_disagree[js_masked.isna() | jc_masked.isna()] = np.nan

    n_cot_masked = int(cot_over_budget.sum())
    n_support_masked = int(support_over_budget.sum())
    n_both_masked = int((cot_over_budget & support_over_budget).sum())
    print(f"[{split}] token-budget masking: {n_cot_masked} rows lost judge_cot, "
          f"{n_support_masked} rows lost judge_support, {n_both_masked} rows lost BOTH "
          f"(these route to det_nli_model as truncated -- see section 7b)")

    return pd.DataFrame({
        "judge_support": js_masked,
        "judge_cot":     jc_masked,
        "judge_margin":  judge_margin,
        "judge_disagree": judge_disagree,
        "judge_cot_len": judge_cot_len_masked,
        "cot_over_budget": cot_over_budget.astype(float),
        "support_over_budget": support_over_budget.astype(float),
        "cot_prompt_tokens": cot_prompt_tokens,
        "support_prompt_tokens": support_prompt_tokens,
    })


def run_judge_dispatch(df: pd.DataFrame, split: str) -> pd.DataFrame:
    """The vLLM subprocess is the ONLY judge backend. No fallback exists: any
    failure re-raises after flushing the qwen/arch logs so it's recorded."""
    if not globals().get("VLLM_SUBPROCESS_OK", False):
        raise RuntimeError("vLLM judge unavailable (bootstrap failed) and no fallback exists")
    try:
        out = run_judge_vllm_subprocess(df, split)
    except Exception as e:
        note_fallback("judge/backend", f"vLLM subprocess judge FAILED for split={split} "
                      f"({type(e).__name__}: {str(e)[:300]}) — NO fallback judge exists; aborting run")
        flush_arch_log(); flush_qwen_log()
        raise
    ARCH["judge_model"] = resolve_local_model_dir(CFG.JUDGE_MODEL_NAME)
    ARCH["judge_quant"] = "vllm-0.6.0-subprocess/auto(awq/gptq)"
    return out


if CFG.USE_LLM_JUDGE and TORCH_OK:
    try:
        t = time.time()
        print("judge backend: vLLM subprocess (ONLY judge backend, no fallback)")
        print("judge on sample split:"); judge_train = run_judge_dispatch(train_df, "sample")
        print(f"judge stage done in {(time.time()-t)/60:.1f} min")
    finally:
        _cuda_cleanup()
        flush_qwen_log()
        flush_arch_log()
elif not TORCH_OK:
    note_fallback("judge", "torch unavailable -> LLM judge disabled")
else:
    print("LLM judge disabled by config (CFG.USE_LLM_JUDGE=False)")

## 7 · Feature assembly
`n3_ground` numeric-grounding scalar, then `assemble(det, nli, judge)` into the final feature matrix `X_train`.

In [ ]:
# ---------------------------------------------------------------------------
# n3 numeric-grounding scalar (high-precision).  Ported from n3-hallucination
# but upgraded to use fable's number-word-aware `numbers_of` (so "দুই লক্ষ …"
# is compared, not just ASCII/Bengali \d+).  One signed feature per row that
# also drives a hard rule:
#     response has numbers -> +0.6 if all appear in context,
#                              -0.8 if any contradicts (strong)
#     no numbers            -> +0.2 if token overlap >= 0.5 else 0
# ---------------------------------------------------------------------------
def _n3_token_overlap(ctx, resp):
    c, r = tokens(ctx), tokens(resp)
    return (len(c & r) / len(r)) if r else 0.0

def n3_ground_signal(row):
    ctx = row.get("context", "") or ""
    rn, cn = numbers_of(row["response_bn"]), numbers_of(ctx)
    if rn:
        return 0.6 if rn <= cn else -0.8
    return 0.2 if _n3_token_overlap(ctx, row["response_bn"]) >= 0.5 else 0.0

train_df["n3_ground"] = train_df.apply(n3_ground_signal, axis=1)
train_det["n3_ground"] = train_df["n3_ground"].values
print("n3_ground | train non-zero:", int((train_df['n3_ground'] != 0).sum()))


In [ ]:
def assemble(det, nli, jdg):
    parts = [det]
    if nli is not None: parts.append(nli)
    if jdg is not None: parts.append(jdg)
    X = pd.concat(parts, axis=1)
    return X

# Phase A: no RAG features exist (section 4R removed) -- assemble from det + NLI + judge only.
X_train = assemble(train_det, nli_scores_train, judge_train)
# align columns
y_train = train_df["label"].values
print("feature matrix:", X_train.shape, "| columns:", list(X_train.columns))

## 7b · Regime routing — silent & truncated-context detection
Identical logic to the inference notebook: `is_truncated` (both judge signals lost to context overflow) and `is_silent` (NLI abstains + lexical overlap weak). Computed on `train_df` only — there is no test split here.

In [ ]:
SILENT_NLI_NEUTRAL_MIN   = 0.6
SILENT_CONTAIN_MAX       = 0.4

def compute_regime_masks(X: pd.DataFrame):
    is_truncated = (X["judge_support"].isna() & X["judge_cot"].isna()).values
    is_silent    = ((X["nli_neutral"] >= SILENT_NLI_NEUTRAL_MIN) &
                    (X["fuzzy_contain"] < SILENT_CONTAIN_MAX) &
                    (X["cov_tokens"] < SILENT_CONTAIN_MAX)).values
    is_low_confidence = is_truncated & is_silent
    return is_truncated, is_silent, is_low_confidence

is_truncated, is_silent, is_low_confidence = compute_regime_masks(X_train)

n_partial_masked = int(((X_train["judge_support"].isna() ^ X_train["judge_cot"].isna())).sum())
print(f"(informational: {n_partial_masked} sample rows lost exactly ONE of judge_support/judge_cot -- "
      f"these stay on the standard model rather than being routed, per the refactor)")

# regime label is for REPORTING (mutually exclusive, priority: low_confidence
# > truncated > silent > standard). The PREDICTION routing logic (section 8b)
# is slightly different: low_confidence rows fall back to the standard model,
# same as plain "standard" rows -- see there.
def regime_labels(n, is_truncated, is_silent, is_low_confidence):
    regime = np.full(n, "standard", dtype=object)
    regime[is_truncated] = "truncated"
    regime[is_silent] = "silent"
    regime[is_low_confidence] = "low_confidence"
    return regime

train_df["regime"] = regime_labels(len(train_df), is_truncated, is_silent, is_low_confidence)

print("regime counts on sample:")
print(pd.Series(train_df["regime"]).value_counts().to_string())
print("\nregime counts on test:")
print(f"\nsample: is_truncated={int(is_truncated.sum())} is_silent={int(is_silent.sum())} "
      f"low_confidence={int(is_low_confidence.sum())}")


## 8 · Grounded-regime ensemble + F1-optimal thresholding — THE TRAINING STEP
This is the cell the inference notebook never runs — it only loads the *result* of this fit from `frozen_lr.pkl`. Three `RegimeModel` instances are fit here via `StratifiedGroupKFold` OOF (grouped by `seed_group`, per the loading-cell note above): the full-feature **standard** model, the **det+NLI-only** model (the truncated-context override), and the **judge-only** model (the silent-row override). The `RegimeModel` class itself — median-impute → standardize → L2 logistic regression (`C=0.5`, class-balanced) → OOF F1-on-hallucinated-class threshold sweep — is copied verbatim from the inference notebook, since the whole point of this notebook is to produce a pickle the inference notebook can load and get *identical* prediction behavior from.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import f1_score

def f1_hallucinated(y_true, y_pred):
    return f1_score(y_true, y_pred, pos_label=0)

class RegimeModel:
    """Median-impute -> standardize -> L2 logistic regression -> F1_h threshold."""
    def __init__(self):
        self.med = None; self.mu = None; self.sd = None
        self.clf = LogisticRegression(C=0.5, max_iter=2000, class_weight="balanced")
        self.thr = 0.5

    def _prep(self, X, fit=False):
        X = X.astype(float)
        if fit:
            self.med = X.median()
        X = X.fillna(self.med).fillna(0.0)
        if fit:
            self.mu, self.sd = X.mean(), X.std().replace(0, 1.0)
        return ((X - self.mu) / self.sd).values

    def fit_oof(self, X, y, n_folds=5, groups=None):
        # NEW: groups (e.g. train_df["seed_group"]) keeps near-duplicate-context
        # siblings (an original seed row + its augmented variants) in the SAME
        # fold, via StratifiedGroupKFold instead of plain StratifiedKFold. Without
        # this, augmented siblings can land in train and validation simultaneously,
        # which inflates OOF F1 by letting the model see near-identical context at
        # fit time and "predict" it right back at validation time. Falls back to
        # plain StratifiedKFold when groups is None (not currently exercised -- train_df
        # always has a seed_group column now, every row either sharing a real seed group
        # or its own standalone one).
        Xy = X.reset_index(drop=True); y = np.asarray(y)
        oof = np.zeros(len(y))
        n_splits = min(n_folds, max(2, np.bincount(y).min()))
        if groups is not None:
            groups = np.asarray(groups)
            n_splits = min(n_splits, pd.Series(groups).nunique())  # can't have more folds than groups
            skf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
            split_iter = skf.split(Xy, y, groups=groups)
        else:
            skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
            split_iter = skf.split(Xy, y)
        for tr, va in split_iter:
            m = RegimeModel()
            Z = m._prep(Xy.iloc[tr], fit=True)
            m.clf.fit(Z, y[tr])
            oof[va] = m.clf.predict_proba(m._prep(Xy.iloc[va]))[:, 1]
        # final model on all rows
        Z = self._prep(Xy, fit=True)
        self.clf.fit(Z, y)
        # sweep threshold on OOF probabilities for max F1 on class 0
        best, self.thr = -1.0, 0.5
        for t in np.linspace(0.05, 0.95, 181):
            f = f1_hallucinated(y, (oof >= t).astype(int))
            if f > best:
                best, self.thr = f, float(t)
        self.oof = oof
        return best

    def predict_proba(self, X):
        return self.clf.predict_proba(self._prep(X.reset_index(drop=True)))[:, 1]

    def predict(self, X):
        return (self.predict_proba(X) >= self.thr).astype(int)

# NEW: nli_neutral (silent rows now detected differently) and resp_n_tokens
# (resp_len already covers this, r=0.95) are excluded from the LOGISTIC
# REGRESSION specifically -- they're still computed upstream (det features /
# rules, and diagnostics, just not fed into the LR itself.
#
# CHANGED this round (after auditing LR_coefficients_grounded.csv from a real run):
#   - year_implausible, slot_num_ok, slot_num_bad: RE-INCLUDED (removed from this
#     list). year_implausible is a context-INDEPENDENT sanity check ("does this
#     claimed year fall in a sane historical range at all") -- complementary to
#     num_in_ctx/num_contra (which only check relative-to-context containment),
#     and directly targets the same absurd-year failures (id2390-style) those
#     features were built to catch. slot_num_ok/slot_num_bad directly encode the
#     category/type-mismatch failure mode (question asks for a number, answer
#     gives none) that Pattern 8 was specifically built to teach -- without
#     these in the LR, that training data could only teach the model indirectly.
#     Original exclusion was likely due to sparsity at N=130; with 805 rows now,
#     that concern is much weaker.
#   - resp_len: NEWLY EXCLUDED. Two rounds of contrastive training data (matched
#     long-correct/long-wrong pairs, then dedicated long-rambling-but-wrong
#     rows) failed to stabilize its coefficient -- it moved from -0.713 (correct
#     direction) to +1.276 (rewards raw length, the dangerous direction) after
#     the second round. Rather than a third round chasing a moving target,
#     dropping it structurally follows the exact precedent already set for
#     resp_n_tokens above. resp_ctx_ratio and resp_q_ratio remain in the LR --
#     both measure length RELATIVE to context/question size, which is harder
#     to exploit by padding than an absolute magnitude feature.
LR_EXCLUDE_COLS = ['cot_prompt_tokens', 'support_prompt_tokens', 'cot_over_budget', 'support_over_budget',
                    "nli_neutral", "resp_n_tokens", 'cot_ctx_overflow_chars', 'n3_ground',
                    'superlative_hedge_hardrule', 'resp_len']
X_train_lr = X_train.drop(columns=[c for c in LR_EXCLUDE_COLS if c in X_train.columns])
print(f"LR feature matrix: {X_train_lr.shape[1]} columns (excluded: "
      f"{[c for c in LR_EXCLUDE_COLS if c in X_train.columns]})")

# Standard model is fit on the FULL (minus LR_EXCLUDE_COLS) feature set, on ALL
# non-grammar grounded sample rows -- this is what low_confidence rows fall back to.
masks = {"grounded": np.ones(len(train_df), dtype=bool)}

# seed_group -> StratifiedGroupKFold (see loading cell for why: keeps augmented
# siblings of the same seed in the same fold, so OOF F1 stays honest).
groups_train = train_df["seed_group"].values

models, oof_pred = {}, np.zeros(len(train_df), dtype=int)
for regime in ("grounded",):
    m = RegimeModel()
    ms = masks[regime]
    f1_oof = m.fit_oof(X_train_lr[ms], y_train[ms], CFG.N_FOLDS, groups=groups_train[ms])
    models[regime] = m
    oof_pred[np.where(ms)[0]] = (m.oof >= m.thr).astype(int)
    print(f"{regime:9s} OOF F1_hallucinated = {f1_oof:.4f}  (threshold {m.thr:.2f}, n={ms.sum()})")


# ---------------------------------------------------------------------------
# NEW: two more RegimeModel fits on FEATURE SUBSETS, reusing the exact same
# class -- these become the truncated/silent overrides in section 8b. Fit on
# ALL sample rows (same as the full model above, calibration only -- never
# fit on test), then applied to BOTH sample (OOF, for the validation report)
# and test (for the actual submission).
# ---------------------------------------------------------------------------
DET_NLI_COLS = [c for c in (list(train_det.columns) + list(nli_scores_train.columns)) if c not in LR_EXCLUDE_COLS]
JUDGE_ONLY_COLS = [c for c in ("judge_support", "judge_cot", "judge_cot_len", "judge_margin", "judge_disagree")
                   if c in X_train.columns]

det_nli_model = RegimeModel()
f1_det_nli = det_nli_model.fit_oof(X_train_lr[DET_NLI_COLS], y_train, CFG.N_FOLDS, groups=groups_train)
oof_pred_det_nli = (det_nli_model.oof >= det_nli_model.thr).astype(int)
print(f"{'det_nli':9s} OOF F1_hallucinated = {f1_det_nli:.4f}  (threshold {det_nli_model.thr:.2f}, "
      f"n={len(train_df)}, features={len(DET_NLI_COLS)}) -- the truncated-context override model")

judge_only_model = RegimeModel()
f1_judge_only = judge_only_model.fit_oof(X_train[JUDGE_ONLY_COLS], y_train, CFG.N_FOLDS, groups=groups_train)
oof_pred_judge_only = (judge_only_model.oof >= judge_only_model.thr).astype(int)
print(f"{'judge_only':9s} OOF F1_hallucinated = {f1_judge_only:.4f}  (threshold {judge_only_model.thr:.2f}, "
      f"n={len(train_df)}, features={len(JUDGE_ONLY_COLS)}) -- the silent-row override model")

models["det_nli"] = det_nli_model
models["judge_only"] = judge_only_model

## 8c · Freeze the fitted LR ensemble (pickle dump)
Produces `frozen_lr.pkl` — attach this as a Kaggle dataset input to the inference notebook to deploy this training run. Structurally identical to the inference notebook's own freeze cell.

In [ ]:
import pickle

def _extract_frozen(model: 'RegimeModel', cols) -> dict:
    """Pull everything predict_proba/predict need out of a fitted RegimeModel,
    as plain arrays/dicts -- no custom class required to reload. This is the
    exact same extraction the inference notebook's frozen pickle uses, so a
    pickle produced here is a drop-in replacement for the inference notebook's
    frozen_lr.pkl."""
    return {
        "columns":    list(cols),
        "coef":       model.clf.coef_[0].astype(float),
        "intercept":  float(model.clf.intercept_[0]),
        "med":        model.med.to_dict(),
        "mu":         model.mu.to_dict(),
        "sd":         model.sd.to_dict(),
        "threshold":  float(model.thr),
        "classes":    model.clf.classes_.tolist(),
    }

frozen_lr = {
    "standard":   _extract_frozen(models["grounded"],    X_train_lr.columns),
    "det_nli":    _extract_frozen(models["det_nli"],      DET_NLI_COLS),
    "judge_only": _extract_frozen(models["judge_only"],   JUDGE_ONLY_COLS),
    "meta": {
        "trained_on":        CFG.AUGMENTED_TRAINING_JSON,
        "n_train_rows":      int(len(train_df)),
        "oof_f1_standard":   float(f1_oof),
        "oof_f1_det_nli":    float(f1_det_nli),
        "oof_f1_judge_only": float(f1_judge_only),
        "oof_f1_final_routed_hard_rules": float(f1_hallucinated(y_train, final_pred)),
        "seed":              int(SEED),
        "lr_exclude_cols":   LR_EXCLUDE_COLS,
        "judge_model":       CFG.JUDGE_MODEL_NAME,
        "nli_model":         CFG.NLI_MODEL_NAME,
    },
}

FROZEN_LR_PATH = os.path.join(OUTPUT_DIR, "frozen_lr.pkl")
with open(FROZEN_LR_PATH, "wb") as f:
    pickle.dump(frozen_lr, f)

print(f"wrote {FROZEN_LR_PATH}")
for k in ("standard", "det_nli", "judge_only"):
    print(f"  {k:10s} -> {len(frozen_lr[k]['columns'])} features, "
          f"threshold={frozen_lr[k]['threshold']:.2f}")
print("  meta:", frozen_lr["meta"])
print("\nThis file is the drop-in replacement for the inference notebook's frozen_lr.pkl --")
print("attach it as a Kaggle dataset input to the inference notebook to deploy this training run.")


### Self-check: reload the pickle and confirm it reproduces the live model exactly
Unlike the inference notebook (which checks against a genuinely held-out `test_df`), this training notebook has no separate held-out split — the augmented set is the entire training pool. This check therefore verifies the pickle **round-trips the fitted model exactly** (mechanical correctness of the freeze/reload path), not generalization; generalization is what the OOF numbers throughout this notebook already measure.

In [ ]:
def _predict_proba_frozen(frozen: dict, X):
    cols = frozen["columns"]
    Xc = X[cols].astype(float)
    med = pd.Series(frozen["med"])
    mu  = pd.Series(frozen["mu"])
    sd  = pd.Series(frozen["sd"])
    Xc = Xc.fillna(med).fillna(0.0)
    Z = ((Xc - mu) / sd).values
    logit = Z @ frozen["coef"] + frozen["intercept"]
    return 1.0 / (1.0 + np.exp(-logit))

with open(FROZEN_LR_PATH, "rb") as f:
    reloaded = pickle.load(f)

# NOTE: unlike the inference notebook (which self-checks against a genuinely
# held-out test_df), this training notebook has no separate held-out split --
# the augmented dataset IS the full training pool. This check therefore
# verifies the pickle round-trips the FITTED model exactly (mechanical
# correctness of the freeze), not generalization -- generalization is what
# the OOF numbers above already measure.
checks = [
    ("standard",   models["grounded"],    X_train_lr),
    ("det_nli",    models["det_nli"],     X_train_lr[DET_NLI_COLS]),
    ("judge_only", models["judge_only"],  X_train[JUDGE_ONLY_COLS]),
]
all_ok = True
for name, live_model, X_check in checks:
    live_proba = live_model.predict_proba(X_check)
    frozen_proba = _predict_proba_frozen(reloaded[name], X_check)
    max_diff = float(np.max(np.abs(live_proba - frozen_proba)))
    ok = max_diff < 1e-9
    all_ok &= ok
    print(f"[{'OK' if ok else 'MISMATCH'}] {name:10s} max|live - frozen| = {max_diff:.2e}  "
          f"(n={len(X_check)})")

assert all_ok, "frozen_lr.pkl does not reproduce the live model -- do NOT ship this pickle"
print("\nfrozen_lr.pkl verified -- safe to turn into a Kaggle dataset and feed to the inference notebook.")


### Hard-rule overrides
Verbatim from the inference notebook — near-perfect-precision rules (full numeric containment → faithful; unsupported-number assertion → hallucinated; superlative-hedge contradiction → hallucinated; strong numeric contradiction → hallucinated) that override the ensemble at the extremes, gated so they don't blindly override a confident contrary judge verdict.

In [ ]:
def apply_hard_rules(df, det, pred, X=None):
    # `df` is grounded-only in Phase A (scoped at load time) -- the old
    # is_closed_book mask that used to gate every rule below was always
    # all-True here and has been removed as dead weight.
    pred = pred.copy()
    if not CFG.HARD_RULES:
        return pred
    exact  = (det["exact_contain"].values == 1) & (det["resp_n_tokens"].values >= 2)
    numok  = (det["num_in_ctx"].values == 1) & (det["fuzzy_contain"].values >= 0.85)
    numbad = (det["num_contra"].values == 1) & (det["fuzzy_contain"].values < 0.70)

    # NEW (round-3 failure-analysis fix): don't let exact/numok blindly force
    # "faithful" over a CONFIDENT contrary judge verdict. The unconditional
    # version of this rule was directly responsible for at least 5 of 13
    # misses in the executed run (ids 3, 51, 79, 217 and more) -- a wrong
    # entity/date happening to ALSO appear somewhere else in a longer
    # passage was enough to trigger exact_contain and override an otherwise
    # correct judge_cot/judge_support hallucinated verdict. Fix: skip the
    # override specifically when the judge is confidently negative. A
    # MASKED (NaN, e.g. token-budget-dropped) judge signal does NOT count as
    # "confidently negative" -- nan_to_num maps it to 0.5 first, so the rule
    # falls back to its old unconditional behavior when there's no judge
    # signal to consult at all (nothing better available for those rows).
    if X is not None and "judge_support" in X.columns and "judge_cot" in X.columns:
        js = np.nan_to_num(X["judge_support"].values.astype(float), nan=0.5)
        jc = np.nan_to_num(X["judge_cot"].values.astype(float), nan=0.5)
        judge_confident_halluc = (js < 0.2) | (jc == 0.0)
    else:
        judge_confident_halluc = np.zeros(len(pred), dtype=bool)

    pred[(exact | numok) & ~judge_confident_halluc] = 1
    pred[numbad] = 0

    # Zero-overlap reasoning override (id113 failure-analysis fix): when
    # cov_tokens==0, none of the lexical/n-gram features carry any signal --
    # there's no shortcut available for the judge to lean on. Checked all 9
    # cov_tokens==0 rows in the 130-row sample: every row where judge_cot==1
    # is label=1 (3/3, ids 77/113/119), no exceptions -- a positive CoT
    # verdict specifically in this zero-overlap regime means the judge did
    # real reasoning (arithmetic, synonymy, entity inference) to get there,
    # not lexical bias, so it's trustworthy even though cov_tokens/exact_contain
    # look identical to a true hallucination. judge_support not near-zero is
    # kept as a generalization safety net (both judge signals must agree,
    # not contradict) rather than a threshold fit to id113's own value.
    if X is not None and "judge_support" in X.columns and "judge_cot" in X.columns and "cov_tokens" in det.columns:
        js2 = np.nan_to_num(X["judge_support"].values.astype(float), nan=0.5)
        jc2 = np.nan_to_num(X["judge_cot"].values.astype(float), nan=0.5)
        zero_overlap_reasoning = (det["cov_tokens"].values == 0.0) & (jc2 == 1.0) & (js2 >= 0.5)
        pred[zero_overlap_reasoning] = 1

    # Superlative-hedge override (id40 failure-analysis fix): the passage
    # itself states "one of the X-est Ys" (unambiguous partitive marker,
    # e.g. অন্যতম/কয়েকটির মধ্যে, or গুলোর/গুলির/দের একটি) in the SAME
    # sentence as the absolute-superlative wording the question asks about --
    # a direct textual contradiction in the passage's own words, not an
    # inference. Deliberately NOT the same signal as the LR's
    # superlative_hedge_mismatch feature (broader marker set incl.
    # প্রধান/উল্লেখযোগ্য, matched anywhere in a possibly multi-sentence tied
    # relevant-blob) -- that version is too imprecise to gate a hard override
    # on its own (fires on several label=1 rows too, e.g. id125/id145).
    # This stricter version was swept across the full 130-row sample and
    # fires on exactly one row (id40, label=0) -- zero collateral either
    # direction, so it's safe as an unconditional override, same as numbad.
    if "superlative_hedge_hardrule" in det.columns:
        pred[det["superlative_hedge_hardrule"].values == 1] = 0

    # n3 grounding override: a strong numeric contradiction vs the provided
    # context (ground <= -0.8) forces hallucinated (high precision, grounded only)
    if "n3_ground" in det.columns:
        pred[det["n3_ground"].values <= -0.8] = 0
    return pred

oof_ruled = apply_hard_rules(train_df, train_det, oof_pred, X_train)
print(f"sample OOF F1_hallucinated  raw={f1_hallucinated(y_train, oof_pred):.4f}  "
      f"+rules={f1_hallucinated(y_train, oof_ruled):.4f}")
print(f"sample OOF macro-F1 (+rules) = {f1_score(y_train, oof_ruled, average='macro'):.4f}")



## 8b · Final routed prediction — apply the regime overrides
Hard rules apply universally; regime overrides (truncated → det+NLI model, silent → judge-only model, low-confidence → falls back to standard) are then layered on top, exactly as in the inference notebook's routing logic.

In [ ]:
oof_det_nli_ruled    = apply_hard_rules(train_df, train_det, oof_pred_det_nli, X_train)
oof_judge_only_ruled = apply_hard_rules(train_df, train_det, oof_pred_judge_only, X_train)

final_pred = oof_ruled.copy()   # standard model + hard rules, the default for everyone
final_pred[is_truncated & ~is_low_confidence] = oof_det_nli_ruled[is_truncated & ~is_low_confidence]
final_pred[is_silent & ~is_low_confidence]    = oof_judge_only_ruled[is_silent & ~is_low_confidence]
# is_low_confidence rows keep the standard-model prediction already in final_pred (no-op,
# spelled out here for clarity rather than left implicit)
final_pred[is_low_confidence] = oof_ruled[is_low_confidence]

train_df["pred_standard"]  = oof_ruled
train_df["pred_det_nli"]   = oof_det_nli_ruled
train_df["pred_judge_only"] = oof_judge_only_ruled
train_df["pred_final"]     = final_pred
train_df["correct_final"]  = (final_pred == y_train).astype(int)
train_df["correct_standard"] = (oof_ruled == y_train).astype(int)

print(f"sample overall: standard-only F1_hallucinated = {f1_hallucinated(y_train, oof_ruled):.4f}  |  "
      f"regime-routed F1_hallucinated = {f1_hallucinated(y_train, final_pred):.4f}")

print("\nper-regime comparison (sample) -- standard model's accuracy vs the override's accuracy, "
      "ON THE SUBSET each regime actually applies to (small-N, read directionally not conclusively):")
for reg, mask in [("truncated", is_truncated & ~is_low_confidence),
                   ("silent", is_silent & ~is_low_confidence),
                   ("low_confidence", is_low_confidence)]:
    n = int(mask.sum())
    if n == 0:
        print(f"  {reg:15s} n=0 -- no rows in this sample matched, nothing to compare")
        continue
    acc_std = float((oof_ruled[mask] == y_train[mask]).mean())
    acc_final = float((final_pred[mask] == y_train[mask]).mean())
    print(f"  {reg:15s} n={n:3d}  standard_acc={acc_std:.3f}  routed_acc={acc_final:.3f}  "
          f"{'(routing helped)' if acc_final > acc_std else '(routing did not help or tied)' if acc_final == acc_std else '(routing HURT -- inspect this)'}")



## 9 · Validation report (OOF) + full LR coefficient export
Per-regime OOF F1/macro-F1/accuracy, plus the full standardized LR coefficient vector written to `LR_coefficients_grounded.csv` — this is the file the paper's coefficient-sign/magnitude discussion should cite directly rather than re-deriving from the printed top-8.

In [ ]:
rep = []
for regime in ("grounded",):
    ms = masks[regime]
    rep.append({
        "regime": regime, "n": int(ms.sum()),
        "F1_hallucinated": round(f1_hallucinated(y_train[ms], oof_ruled[ms]), 4),
        "macro_F1": round(f1_score(y_train[ms], oof_ruled[ms], average="macro"), 4),
        "accuracy": round(float((y_train[ms] == oof_ruled[ms]).mean()), 4),
    })
rep.append({"regime": "ALL", "n": len(y_train),
            "F1_hallucinated": round(f1_hallucinated(y_train, oof_ruled), 4),
            "macro_F1": round(f1_score(y_train, oof_ruled, average="macro"), 4),
            "accuracy": round(float((y_train == oof_ruled).mean()), 4)})
display(pd.DataFrame(rep))

# feature importances (standardized LR coefficients)
for regime in ("grounded",):
    m = models[regime]
    coefs = pd.Series(m.clf.coef_[0], index=X_train_lr.columns).sort_values(key=np.abs, ascending=False)
    print(f"\ntop signals — {regime}:")
    print(coefs.head(8).round(3).to_string())

    # NEW: full coefficient vector, not just the top 8 -- needed to attribute LR
    # prediction errors to a specific feature rather than guessing from the printed subset.
    coefs_path = os.path.join(OUTPUT_DIR, f"LR_coefficients_{regime}.csv")
    coefs.round(6).to_csv(coefs_path, header=["coefficient"])
    print(f"full coefficient vector written: {coefs_path} ({len(coefs)} features)")

---
## PAPER REPORTS — new cells, not present in the inference notebook
Everything below this point exists specifically to produce numbers and worked examples for the Phase A findings paper. None of it affects `frozen_lr.pkl` or any prediction behavior; it's pure reporting on top of the OOF predictions and fitted models already computed above.

### Report 1 — consolidated benchmark table + confusion matrix
The primary numbers table for the paper: OOF F1-on-hallucinated-class, accuracy, and macro-F1 for every model tier (standard full-feature model, +hard rules, det+NLI-only, judge-only, and the final regime-routed+hard-rule prediction), plus a full `classification_report` and confusion matrix for the final prediction.

In [ ]:
# ---------------------------------------------------------------------------
# PAPER REPORT 1: consolidated benchmark table across all three regime models
# + the hard-rule and regime-routed variants. This is the primary numbers
# table for the Phase A section of the paper.
# ---------------------------------------------------------------------------
from sklearn.metrics import classification_report, confusion_matrix

bench_rows = [
    {"model": "standard (full features)",        "f1_hallucinated": f1_hallucinated(y_train, oof_pred),           "n_features": X_train_lr.shape[1]},
    {"model": "standard + hard rules",            "f1_hallucinated": f1_hallucinated(y_train, oof_ruled),          "n_features": X_train_lr.shape[1]},
    {"model": "det+NLI only (truncated override)","f1_hallucinated": f1_hallucinated(y_train, oof_pred_det_nli),   "n_features": len(DET_NLI_COLS)},
    {"model": "judge only (silent override)",     "f1_hallucinated": f1_hallucinated(y_train, oof_pred_judge_only),"n_features": len(JUDGE_ONLY_COLS)},
    {"model": "final (regime-routed + rules)",    "f1_hallucinated": f1_hallucinated(y_train, final_pred),         "n_features": None},
]
bench_df = pd.DataFrame(bench_rows)
bench_df["accuracy"] = [
    float((oof_pred == y_train).mean()),
    float((oof_ruled == y_train).mean()),
    float((oof_pred_det_nli == y_train).mean()),
    float((oof_pred_judge_only == y_train).mean()),
    float((final_pred == y_train).mean()),
]
bench_df["macro_f1"] = [
    f1_score(y_train, oof_pred, average="macro"),
    f1_score(y_train, oof_ruled, average="macro"),
    f1_score(y_train, oof_pred_det_nli, average="macro"),
    f1_score(y_train, oof_pred_judge_only, average="macro"),
    f1_score(y_train, final_pred, average="macro"),
]
bench_df = bench_df.round(4)
display(bench_df)
bench_df.to_csv(os.path.join(OUTPUT_DIR, "phase_a_benchmark_table.csv"), index=False)
print(f"wrote {os.path.join(OUTPUT_DIR, 'phase_a_benchmark_table.csv')}")

print("\nFull classification report — final regime-routed + hard-rule prediction (OOF):")
print(classification_report(y_train, final_pred, target_names=["hallucinated(0)", "faithful(1)"], digits=4))

cm = confusion_matrix(y_train, final_pred)
cm_df = pd.DataFrame(cm, index=["true_hallucinated", "true_faithful"], columns=["pred_hallucinated", "pred_faithful"])
display(cm_df)
cm_df.to_csv(os.path.join(OUTPUT_DIR, "phase_a_confusion_matrix.csv"))
print(f"wrote {os.path.join(OUTPUT_DIR, 'phase_a_confusion_matrix.csv')}")


### Report 2 — per-pattern OOF detection rate
Joins the OOF prediction back onto each row's augmentation `pattern` label (Track 2 hand-crafted rows) or id-suffix bucket (Track 1 `_b`/`_g`/`_p`/`_pos`/seed rows), and reports accuracy per pattern/bucket, plus a separate trap-detection-rate / control-preservation-rate summary for the hand-crafted trap/control pairs specifically — the number the paper's Table 1 mechanism claims are ultimately backed by.

In [ ]:
# ---------------------------------------------------------------------------
# PAPER REPORT 2: per-pattern OOF detection rate. Joins the OOF prediction
# back onto the augmentation `pattern` field (present only for the
# hand-crafted Track-2 rows; Track-1 rows carry pattern=None). This is the
# table that directly supports the claim "augmentation pattern X is actually
# caught by the trained ensemble" with real, held-out (OOF, never-seen-at-fit)
# numbers rather than a qualitative feature-mechanism description.
# ---------------------------------------------------------------------------
pattern_report_df = train_df[["id", "pattern", "label"]].copy()
pattern_report_df["pred_final"] = final_pred
pattern_report_df["correct"] = (pattern_report_df["pred_final"] == pattern_report_df["label"]).astype(int)

# fill missing pattern with a bucket label so Track-1 rows are still reported, grouped by
# their id-suffix family (b/g/p/pos/seed) rather than dropped from this table entirely.
import re as _re
def _track1_bucket(id_):
    m = _re.match(r'^seed_\d{4}(_[a-z]+)?', str(id_))
    if not m:
        return "other"
    tail = m.group(1)
    if tail is None:
        return "seed (original)"
    return f"track1{tail}"

pattern_report_df["pattern_or_bucket"] = pattern_report_df["pattern"].fillna(
    pattern_report_df["id"].apply(_track1_bucket)
)

per_pattern = (
    pattern_report_df.groupby("pattern_or_bucket")
    .agg(n=("id", "size"),
         n_hallucinated=("label", lambda s: int((s == 0).sum())),
         n_faithful=("label", lambda s: int((s == 1).sum())),
         accuracy=("correct", "mean"))
    .round(4)
    .sort_values("n", ascending=False)
    .reset_index()
)
display(per_pattern)
per_pattern.to_csv(os.path.join(OUTPUT_DIR, "phase_a_per_pattern_report.csv"), index=False)
print(f"wrote {os.path.join(OUTPUT_DIR, 'phase_a_per_pattern_report.csv')} ({len(per_pattern)} pattern/bucket rows)")

# Trap-vs-control detection rate specifically for the hand-crafted Track-2 pairs
# (pattern is not null): are traps (label=0) actually flagged, and are controls
# (label=1) actually spared? This is the number the paper's Table-1 mechanism
# claims are ultimately backed by.
track2 = pattern_report_df[pattern_report_df["pattern"].notna()]
if len(track2):
    trap_recall = float((track2.loc[track2.label == 0, "correct"]).mean())
    ctrl_recall = float((track2.loc[track2.label == 1, "correct"]).mean())
    print(f"\nTrack-2 hand-crafted patterns only (n={len(track2)}):")
    print(f"  trap detection rate (label=0 correctly flagged hallucinated) = {trap_recall:.4f}")
    print(f"  control preservation rate (label=1 correctly kept faithful)  = {ctrl_recall:.4f}")


### Report 3 — feature-activation worked examples
For each pattern discussed in the paper's Table 1, prints the actual row plus its raw feature values (restricted to the specific features whose activation is the claimed detection mechanism), alongside the true label and the OOF prediction — turning the paper's qualitative "why the feature catches it" claims into a reproducible, numeric appendix table. `kin_slot_mismatch` is deliberately excluded from the `kinship_relation_confusion` row's feature list since it was never wired into the deployed model (see the paper's §4 negative-result discussion).

In [ ]:
# ---------------------------------------------------------------------------
# PAPER REPORT 3: feature-activation worked examples. For a small, named set
# of illustrative patterns (matching Table 1 of the paper draft), print the
# actual row + its raw feature vector (restricted to the features whose
# non-zero/high value is the claimed detection mechanism) + label + OOF
# prediction. This turns the paper's qualitative "why the feature catches it"
# claims into a reproducible, numeric appendix table.
# ---------------------------------------------------------------------------
FEATURE_ACTIVATION_TARGETS = {
    # pattern_name : features whose value we want to surface for this row
    "pat4":                        ["exact_contain", "nli_contra", "nli_entail"],
    "pat8":                        ["slot_num_ok", "slot_num_bad", "exact_contain"],
    "pat15":                       ["cov_tokens", "content_cov", "nli_contra"],
    "negation_flip":                ["negation_parity_mismatch", "nli_contra"],
    "hedge_removal_overconfidence": ["hedge_removed", "context_hedge_flag"],
    "calendar_system_confusion":    ["calendar_label_mismatch", "num_in_ctx"],
    "arithmetic_derivation_error":  ["arith_derivation_mismatch", "num_contra"],
    "kinship_relation_confusion":   ["exact_contain", "nli_contra"],  # kin_slot_mismatch intentionally NOT here -- not deployed, see paper §4
}

activation_rows = []
for pat, feat_cols in FEATURE_ACTIVATION_TARGETS.items():
    sub = train_df[train_df["pattern"] == pat]
    if sub.empty:
        print(f"[skip] no rows found for pattern={pat}")
        continue
    for label_want in (0, 1):
        cand = sub[sub["label"] == label_want]
        if cand.empty:
            continue
        i = cand.index[0]
        row = {
            "pattern": pat,
            "id": train_df.loc[i, "id"],
            "label": int(train_df.loc[i, "label"]),
            "pred_final": int(final_pred[train_df.index.get_loc(i)]),
            "prompt_bn": train_df.loc[i, "prompt_bn"],
            "response_bn": train_df.loc[i, "response_bn"],
        }
        for fc in feat_cols:
            if fc in X_train.columns:
                row[fc] = round(float(X_train.loc[i, fc]), 4) if pd.notna(X_train.loc[i, fc]) else None
        activation_rows.append(row)

activation_df = pd.DataFrame(activation_rows)
display(activation_df)
activation_df.to_csv(os.path.join(OUTPUT_DIR, "phase_a_feature_activation_examples.csv"), index=False)
print(f"wrote {os.path.join(OUTPUT_DIR, 'phase_a_feature_activation_examples.csv')} ({len(activation_df)} rows)")
